# “Reconstructing long-term nearshore waves by Deep Learning (DL) fedwith physics-based model”

This notebook provides the fully reproducible offshore example associated
with the manuscript:

**“Reconstructing long-term nearshore waves by Deep Learning (DL) fed
with physics-based model”**

The example evaluates the reconstruction of artificial gaps in the public
M1 offshore buoy record using an open SIMAR reference time series.

Three deep-learning architectures are considered:

- GRU/LSTM
- Temporal Convolutional Network (TCN)
- Convolutional Long Short-Term Memory network (ConvLSTM)

The workflow includes data loading, quality control, temporal alignment,
feature engineering, chronological data splitting, model training,
evaluation, reconstruction, and result export.

The intensive nearshore field campaign observations used in the broader
study are not included because they are subject to project-specific
data-management conditions. Publicly releasable nearshore reconstruction
outputs produced with deep learning and SWAN are distributed separately
in the repository.

In [ ]:
from pathlib import Path
import os

# Detect whether the notebook is running on Kaggle.
ON_KAGGLE = Path("/kaggle/input").exists()

if ON_KAGGLE:
    # Kaggle dataset paths.
    BUOY_FILE = Path(
        "/kaggle/input/datasets/mvinesr/data-bv-boya/"
        "Datos_oleaje_PE (1) (1).csv"
    )
    SIMAR_FILE = Path(
        "/kaggle/input/datasets/mvinesr/dataset-simar-31550039/"
        "reference_SIMAR_data.csv"
    )

    # Kaggle's writable working directory.
    OUTPUT_DIR = Path("/kaggle/working/results")

else:
    # Paths used after cloning the GitHub repository.
    PROJECT_ROOT = Path.cwd()

    # Allow execution from either the repository root or notebooks/.
    if PROJECT_ROOT.name in {"notebook", "notebooks"}:
        PROJECT_ROOT = PROJECT_ROOT.parent

    BUOY_FILE = PROJECT_ROOT / "data" / "model_input" / "Wave_data_offshore_M1.csv"
    SIMAR_FILE = (
        PROJECT_ROOT
        / "data"
        / "model_input"
        / "reference_SIMAR_data.csv"
    )
    OUTPUT_DIR = PROJECT_ROOT / "results"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Execution environment: {'Kaggle' if ON_KAGGLE else 'Local/GitHub'}")
print(f"Buoy data: {BUOY_FILE}")
print(f"SIMAR data: {SIMAR_FILE}")
print(f"Output directory: {OUTPUT_DIR}")

if not BUOY_FILE.exists():
    raise FileNotFoundError(f"Buoy dataset not found: {BUOY_FILE}")

if not SIMAR_FILE.exists():
    raise FileNotFoundError(f"SIMAR dataset not found: {SIMAR_FILE}")

In [ ]:
# STEP 1: Import required libraries and configure paths

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.preprocessing.sequence import TimeseriesGenerator
from tensorflow.keras.callbacks import EarlyStopping


# Detect whether the notebook is running on Kaggle
ON_KAGGLE = Path("/kaggle/input").exists()

# Detect whether the notebook is running on Kaggle
ON_KAGGLE = Path("/kaggle/input").exists()

if ON_KAGGLE:

    BUOY_FILE = Path(
        "/kaggle/input/datasets/mvinesr/data-bv-boya/"
        "Datos_oleaje_PE (1) (1).csv"
    )

    SIMAR_FILE = Path(
        "/kaggle/input/datasets/mvinesr/dataset-simar-31550039/"
        "reference_SIMAR_data.csv"
    )

    OUTPUT_DIR = Path("/kaggle/working/results")

else:

    PROJECT_ROOT = Path.cwd()

    # Allow execution from repository root, notebook/, or notebooks/
    if PROJECT_ROOT.name in {"notebook", "notebooks"}:
        PROJECT_ROOT = PROJECT_ROOT.parent

    BUOY_FILE = (
        PROJECT_ROOT
        / "data"
        / "model_input"
        / "Wave_data_offshore_M1.csv"
    )

    SIMAR_FILE = (
        PROJECT_ROOT
        / "data"
        / "model_input"
        / "reference_SIMAR_data.csv"
    )

    OUTPUT_DIR = PROJECT_ROOT / "results"


OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(
    f"Execution environment: "
    f"{'Kaggle' if ON_KAGGLE else 'Local/GitHub'}"
)

if not ON_KAGGLE:
    print(f"Project root: {PROJECT_ROOT}")

print(f"Buoy data: {BUOY_FILE}")
print(f"SIMAR data: {SIMAR_FILE}")
print(f"Output directory: {OUTPUT_DIR}")


# Check that input files exist
if not BUOY_FILE.exists():
    raise FileNotFoundError(
        f"Buoy dataset not found: {BUOY_FILE}"
    )

if not SIMAR_FILE.exists():
    raise FileNotFoundError(
        f"SIMAR dataset not found: {SIMAR_FILE}"
    )


print(
    f"Execution environment: "
    f"{'Kaggle' if ON_KAGGLE else 'Local/GitHub'}"
)
print(f"Buoy data: {BUOY_FILE}")
print(f"SIMAR data: {SIMAR_FILE}")
print(f"Output directory: {OUTPUT_DIR}")


# Load public M1 offshore buoy observations
buoy_df = pd.read_csv(
    BUOY_FILE,
    sep=";"
)

buoy_df["Fecha"] = pd.to_datetime(
    buoy_df["Fecha"],
    dayfirst=True,
    errors="coerce"
)

buoy_df = (
    buoy_df
    .dropna(subset=["Fecha"])
    .set_index("Fecha")
    .sort_index()
)

buoy_df = buoy_df.rename(
    columns={"Hs": "Hs_observed"}
)

buoy_df = buoy_df[["Hs_observed"]]

# Replace invalid buoy values with missing values
buoy_df["Hs_observed"] = buoy_df[
    "Hs_observed"
].replace(-9999.9, np.nan)


# Load public SIMAR reference series
simar_df = pd.read_csv(
    SIMAR_FILE,
    sep=";",
    encoding="latin1"
)

simar_df["Fecha"] = pd.to_datetime(
    simar_df["Fecha"],
    dayfirst=False,
    errors="coerce"
)

simar_df = (
    simar_df
    .dropna(subset=["Fecha"])
    .set_index("Fecha")
    .sort_index()
)

simar_df = simar_df.rename(
    columns={"Hs": "Hs_reference"}
)

simar_df = simar_df[["Hs_reference"]]

In [ ]:
# STEP 2: Inspect the temporal resolution and identify missing timestamps

buoy_series = buoy_df.sort_index()
time_index = buoy_series.index

print("Index dtype:", time_index.dtype)
print("Monotonically increasing:", time_index.is_monotonic_increasing)
print("Inferred frequency:", pd.infer_freq(time_index))

print(
    "Most common time differences:\n",
    time_index.to_series().diff().value_counts().head()
)

# Build the complete expected hourly time index
expected_index = pd.date_range(
    start=time_index.min(),
    end=time_index.max(),
    freq="h"
)

# Identify timestamps that are absent from the original buoy record
missing_timestamps = expected_index.difference(time_index)

print(f"Expected timestamps: {len(expected_index):,}")
print(f"Observed timestamps: {len(time_index):,}")
print(f"Missing timestamps: {len(missing_timestamps):,}")

In [ ]:
buoy_df.index = pd.to_datetime(buoy_df.index)

# Count unique months
num_months = buoy_df.index.to_period("M").nunique()
print("Number of months with data:", num_months)

In [ ]:
buoy_df.index = pd.to_datetime(buoy_df.index)

# Filter between 2012 and 2016
start_date = "2012-01-01"
end_date = "2016-12-31"

buoy_df = buoy_df.loc[
    (buoy_df.index >= start_date) &
    (buoy_df.index <= end_date)
]

# Verify
print(buoy_df)
print(
    "Final temporal range:",
    buoy_df.index.min(),
    "→",
    buoy_df.index.max()
)
print(
    "Number of months with data:",
    buoy_df.index.to_period("M").nunique()
)

In [ ]:
# STEP 3: Clean and align the buoy and SIMAR time series

buoy_df = buoy_df.sort_index()
buoy_df = buoy_df[~buoy_df.index.isna()]
buoy_df = buoy_df[~buoy_df.index.duplicated(keep="first")]

simar_df = simar_df.sort_index()
simar_df = simar_df[~simar_df.index.isna()]
simar_df = simar_df[~simar_df.index.duplicated(keep="first")]

# Target time grid
TARGET_FREQ = "1h"
START = pd.Timestamp("2000-07-05 00:00")
END = pd.Timestamp("2022-02-28 23:00")

time_grid = pd.date_range(
    start=START,
    end=END,
    freq=TARGET_FREQ
)

buoy_df.index = pd.to_datetime(
    buoy_df.index,
    errors="coerce"
)

simar_df.index = pd.to_datetime(
    simar_df.index,
    errors="coerce"
)

buoy_df = buoy_df[
    ~buoy_df.index.isna()
].sort_index()

simar_df = simar_df[
    ~simar_df.index.isna()
].sort_index()

# Ensure numeric columns
buoy_df["Hs_observed"] = pd.to_numeric(
    buoy_df["Hs_observed"],
    errors="coerce"
)

simar_df["Hs_reference"] = pd.to_numeric(
    simar_df["Hs_reference"],
    errors="coerce"
)

# Resample only the required columns
# Missing buoy values are preserved as real gaps
buoy_1h = (
    buoy_df[["Hs_observed"]]
    .resample(TARGET_FREQ)
    .mean()
    .reindex(time_grid)
)

simar_1h = (
    simar_df[["Hs_reference"]]
    .resample(TARGET_FREQ)
    .mean()
    .reindex(time_grid)
)

if pd.infer_freq(buoy_df.index) == TARGET_FREQ:
    buoy_1h = buoy_df.reindex(time_grid)
else:
    buoy_1h = (
        buoy_df
        .resample(TARGET_FREQ)
        .mean()
        .reindex(time_grid)
    )

# Combine both series
df_all = pd.concat(
    [
        buoy_1h[["Hs_observed"]],
        simar_1h[["Hs_reference"]]
    ],
    axis=1
).astype(
    {
        "Hs_observed": "float64",
        "Hs_reference": "float64"
    }
)

print(
    df_all.index.min(),
    "→",
    df_all.index.max(),
    "| frequency:",
    TARGET_FREQ
)

print(
    df_all[
        ["Hs_observed", "Hs_reference"]
    ].head(3)
)

print(
    df_all[
        ["Hs_observed", "Hs_reference"]
    ].tail(3)
)

print(
    "NaNs in buoy series:",
    df_all["Hs_observed"].isna().sum(),
    "| NaNs in reference series:",
    df_all["Hs_reference"].isna().sum()
)

## 1. GRU/LSTM model

This section implements the GRU/LSTM model used to
reconstruct missing significant wave-height observations from the public
offshore buoy record.

The model uses the aligned SIMAR reference series and the corresponding
temporal input sequences to estimate the missing buoy values.

In [ ]:
# -*- coding: utf-8 -*-
# Reconstruction of missing Hs_observed values using LSTM with improvements for extremes
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras import regularizers
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy('float32')   # important
tf.keras.backend.set_floatx('float32')

# =========================================
# 0) Config
# =========================================
np.random.seed(42)
tf.random.set_seed(42)

# Window in 3-hour steps: 56 = 7 days (set to 24 for 3 days)
WINDOW = 56
BATCH  = 64
EPOCHS = 150
P_TRAIN, P_VAL = 0.70, 0.10   


OVERSAMPLE_WEIGHT = 4.5  #  weight assigned to Hs >= training P95
HUBER_DELTA = 1        # in normalized StandardScaler space

# Loss in METRES for the residual using the fold-specific sc_y
def mae_meters_residual_loss(sc_y):
    # y_true and y_pred are in normalized asinh(delta) space
    def loss(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.cast(y_pred, tf.float32)
        # Inverse fold-specific StandardScaler transformation
        y_true_t = y_true * sc_y.scale_[0] + sc_y.mean_[0]
        y_pred_t = y_pred * sc_y.scale_[0] + sc_y.mean_[0]
        # Convert asinh(delta) back to delta in metres
        y_true_m = tf.sinh(y_true_t)
        y_pred_m = tf.sinh(y_pred_t)
        return tf.reduce_mean(tf.abs(y_true_m - y_pred_m))
    return loss

# =========================================
# 1) Temporal selection (adjust the range if required)
# =========================================
# 1) Temporal selection without resampling
df = df_all.loc['1990-01-01':'2026-01-01', ['Hs_observed', 'Hs_reference']].copy().sort_index()

df = df.resample('3h').mean()

# Segments: break at time-step changes or when the reference is missing
deltas = df.index.to_series().diff()
freq   = deltas.mode().iloc[0]
tol    = pd.Timedelta('5min')
same_step = deltas.between(freq - tol, freq + tol)

df['seg'] = (~same_step | df['Hs_reference'].isna()).cumsum().fillna(0).astype(int)


# =========================================
# 2) Chronological temporal split by percentages
# =========================================
def temporal_split_percent(times_index, p_train=0.70, p_val=0.10):
    times_index = pd.DatetimeIndex(times_index)

    if len(times_index) == 0:
        raise ValueError("times_index is empty.")
    if not (0 < p_train < 1 and 0 < p_val < 1 and p_train + p_val < 1):
        raise ValueError("Invalid split percentages.")

    order = np.argsort(times_index.values)
    times_sorted = times_index.values[order]
    n = len(times_sorted)

    i_train_end = int(np.floor(p_train * (n - 1)))
    i_val_end   = int(np.floor((p_train + p_val) * (n - 1)))

    t_train_end = pd.to_datetime(times_sorted[i_train_end])
    t_val_end   = pd.to_datetime(times_sorted[i_val_end])

    m_tr  = (times_index <= t_train_end)
    m_val = (times_index >  t_train_end) & (times_index <= t_val_end)
    m_te  = (times_index >  t_val_end)

    if not (m_tr.any() and m_val.any() and m_te.any()):
        raise ValueError(
            f"Empty split. Counts -> train:{m_tr.sum()}, val:{m_val.sum()}, test:{m_te.sum()}"
        )
    return m_tr, m_val, m_te, t_train_end, t_val_end

# =========================================
# 3) Sequence-to-one window construction
# =========================================
def build_seq2one_feats(arr_feats, target_arr, window, seg_ids):
    X, y, t_idx = [], [], []

    for i in range(window - 1, len(arr_feats)):

        # Do not allow windows to cross temporal segments
        if seg_ids[i] != seg_ids[i - window + 1]:
            continue

        xs = arr_feats[i - window + 1:i + 1, :]

        # Require a complete predictor window
        if np.isnan(xs).any():
            continue

        # Require a valid target
        if np.isnan(target_arr[i]):
            continue

        X.append(xs)
        y.append(target_arr[i])
        t_idx.append(i)

    return (
        np.array(X),
        np.array(y).reshape(-1, 1),
        np.array(t_idx, dtype=np.int64)
    )

def build_features(df):
    out = {}
    dfx = df.copy()

    # Residual target
    dfx['delta']   = dfx['Hs_observed'] - dfx['Hs_reference']
    dfx['delta_t'] = np.arcsinh(dfx['delta'])

    # Base reference without propagation across gaps
    sim_base = dfx['Hs_reference']

    feat = pd.DataFrame(index=dfx.index)
    feat['sim'] = sim_base

    # Rolling features requiring complete windows
    for h in [8, 24, 56]:
        r = sim_base.rolling(h, min_periods=h)
        feat[f'sim_max_{h}']  = r.max()
        feat[f'sim_mean_{h}'] = r.mean()
        feat[f'sim_std_{h}']  = r.std()

    # Lagged features using past values only
    for k in [1, 3, 8, 24, 56]:
        feat[f'sim_lag_{k}'] = sim_base.shift(k)

    # Temporal dynamics
    feat['sim_diff_1'] = sim_base.diff(1)
    feat['sim_diff_3'] = sim_base.diff(3)
    feat['sim_diff_8'] = sim_base.diff(8)
    feat['sim_ema_8']  = sim_base.ewm(span=8,  adjust=False).mean()
    feat['sim_ema_24'] = sim_base.ewm(span=24, adjust=False).mean()
    feat['sim_ema_56'] = sim_base.ewm(span=56, adjust=False).mean()

    

    # Seasonality
    feat['sin_doy']  = np.sin(2*np.pi*feat.index.dayofyear/365.25)
    feat['cos_doy']  = np.cos(2*np.pi*feat.index.dayofyear/365.25)
    feat['sin_hour'] = np.sin(2*np.pi*feat.index.hour/24.0)
    feat['cos_hour'] = np.cos(2*np.pi*feat.index.hour/24.0)
    feat['sin_mon']  = np.sin(2*np.pi*(feat.index.month-1)/12)
    feat['cos_mon']  = np.cos(2*np.pi*(feat.index.month-1)/12)

    out['df'] = dfx
    out['feat_raw'] = feat
    return out




# Walk-forward cross-validation
def make_folds(times, n_folds=3):
    """Generate n_folds with progressively increasing cut-offs over times."""
    n = len(times)
    # Start the first fold near the midpoint to ensure sufficient training data
    idx = np.linspace(int(0.55*n), n-1, n_folds+1, dtype=int)
    folds = []
    for i in range(n_folds):
        i_train_end = idx[i]
        i_val_end   = idx[i+1]
        t_train_end = times[i_train_end]
        t_val_end   = times[i_val_end]
        folds.append((pd.Timestamp(t_train_end), pd.Timestamp(t_val_end)))
    return folds

def build_dataset_with_cuts(df_slice, WINDOW, t_train_end, t_val_end):
    """Leakage-free fold for the residual delta = Hs_observed - Hs_reference in asinh space."""
    dfl = df_slice.copy()
    if 'seg' in df.columns:
            dfl['seg'] = df.loc[dfl.index, 'seg'].to_numpy()
    else:
            deltas = dfl.index.to_series().diff()
            freq   = deltas.mode().iloc[0]
            tol    = pd.Timedelta('5min')
            same_step = deltas.between(freq - tol, freq + tol)
            dfl['seg'] = (~same_step).cumsum().fillna(0).astype(int)

    # Features
    ctx  = build_features(dfl)
    feat = ctx['feat_raw']

    # Residual target in asinh space, required before target scaling
    with np.errstate(invalid='ignore'):
        dfl['delta']   = dfl['Hs_observed'] - dfl['Hs_reference']
        dfl['delta_t'] = np.arcsinh(dfl['delta'])

    # Fit scalers using only the training portion of the fold
    mask_scal = (dfl.index <= t_train_end)          # define before use
    sc_X_f = StandardScaler().fit(feat.loc[mask_scal].values)
    sc_y_f = StandardScaler().fit(dfl.loc[mask_scal, ['delta_t']].dropna())

    feat_sc = pd.DataFrame(sc_X_f.transform(feat.values), index=feat.index, columns=feat.columns)

    dfl['delta_scaled'] = np.nan
    ok = dfl['delta_t'].notna()
    dfl.loc[ok, 'delta_scaled'] = sc_y_f.transform(dfl.loc[ok, ['delta_t']])[:, 0]

    # Windowing: X contains features and y contains delta_scaled
    Xf, yf, t_idx = build_seq2one_feats(
    feat_sc.values, dfl['delta_scaled'].values, WINDOW, dfl['seg'].values
    )
    t_idx = np.asarray(t_idx, dtype=np.int64)
    times_f = dfl.index[t_idx]

    sim_t   = dfl['Hs_reference'].values[t_idx]

    m_tr  = (times_f <= t_train_end)
    m_val = (times_f > t_train_end) & (times_f <= t_val_end)
    return Xf, yf, times_f, m_tr, m_val, sc_y_f, sim_t


def build_dataset_no_leak(df_slice, WINDOW, p_train=P_TRAIN, p_val=P_VAL):
    """Leakage-free global train-validation-test split for the residual formulation."""
    ctx    = build_features(df_slice)
    df_loc = ctx['df']
    feat   = ctx['feat_raw']

    if 'seg' in df.columns:
        df_loc['seg'] = df.loc[df_loc.index, 'seg'].to_numpy()
    else:
        deltas = df_loc.index.to_series().diff()
        freq   = deltas.mode().iloc[0]
        tol    = pd.Timedelta('5min')
        same_step = deltas.between(freq - tol, freq + tol)
        df_loc['seg'] = (~same_step).cumsum().fillna(0).astype(int)


    feat   = ctx['feat_raw']

    # Residual target in asinh space used to determine cut-offs
    df_loc['delta']   = df_loc['Hs_observed'] - df_loc['Hs_reference']
    df_loc['delta_t'] = np.arcsinh(df_loc['delta'])

    # First pass for cut-offs using delta_t
    X0, _, t_idx0 = build_seq2one_feats(feat.values, df_loc['delta_t'].values, WINDOW, df_loc['seg'].values)
    t_idx0 = np.asarray(t_idx0, dtype=np.int64)
    times0 = df_loc.index[t_idx0]
    m_tr0, m_val0, m_te0, t_train_end, t_val_end = temporal_split_percent(times0, p_train, p_val)

    # Fit scalers on training data
    mask_scal = (df_loc.index <= t_train_end)
    sc_X = StandardScaler().fit(feat.loc[mask_scal].values)
    sc_y = StandardScaler().fit(df_loc.loc[mask_scal, ['delta_t']].dropna())

    feat_sc = pd.DataFrame(sc_X.transform(feat.values), index=feat.index, columns=feat.columns)

    df_loc['delta_scaled'] = np.nan
    ok = df_loc['delta_t'].notna()
    df_loc.loc[ok, 'delta_scaled'] = sc_y.transform(df_loc.loc[ok, ['delta_t']])[:, 0]

    X, y, t_idx = build_seq2one_feats(feat_sc.values, df_loc['delta_scaled'].values, WINDOW, df_loc['seg'].values)
    t_idx = np.asarray(t_idx, dtype=np.int64)
    times = df_loc.index[t_idx]
    sim_t = df_loc['Hs_reference'].values[t_idx]

    m_tr, m_val, m_te, t_train_end, t_val_end = temporal_split_percent(times, p_train, p_val)

    return {
        'X': X, 'y': y, 'times': times,
        'm_tr': m_tr, 'm_val': m_val, 'm_te': m_te,
        't_train_end': t_train_end, 't_val_end': t_val_end,
        'sc_X': sc_X, 'sc_y': sc_y,
        'feat_scaled': feat_sc,
        'df_loc': df_loc,
        'sim_t': sim_t
    }


def build_lstm_model(
    n_feats, WINDOW,
    lstm1=48, lstm2=24,
    use_lstm2=True,
    use_gru=True,                 # GRU by default
    pool="gap",
    sp_dropout=0.30,
    rec_dropout=0.10,
    dense_units=16,
    dense_dropout=0.30,
    lr=3e-4,
    l2reg=0.0,
    noise_std=0.00,
    use_adamw=False
):
    reg = tf.keras.regularizers.l2(l2reg) if (l2reg and l2reg > 0) else None
    RNN = tf.keras.layers.GRU if use_gru else tf.keras.layers.LSTM

    inp = tf.keras.layers.Input(shape=(WINDOW, n_feats))
    x = inp
    if noise_std and noise_std > 0:
        x = tf.keras.layers.GaussianNoise(noise_std)(x)

    # Causal convolutional block (light TCN)
    x = tf.keras.layers.Conv1D(32, 3, padding='causal', activation='relu')(x)
    x = tf.keras.layers.Conv1D(32, 5, padding='causal', dilation_rate=2, activation='relu')(x)
    x = tf.keras.layers.SpatialDropout1D(sp_dropout)(x)

    first_return_seq = True if (use_lstm2 or pool == "gap") else False
    x = RNN(
        lstm1,
        return_sequences=first_return_seq,
        recurrent_dropout=rec_dropout,
        kernel_regularizer=reg, recurrent_regularizer=reg
    )(x)
    
    if use_lstm2:
        # With a second layer, retain the sequence only when GAP is used
        second_return_seq = (pool == "gap")
        x = RNN(
            lstm2,
            return_sequences=second_return_seq,
            recurrent_dropout=rec_dropout,
            kernel_regularizer=reg, recurrent_regularizer=reg
        )(x)
    
    # Pooling
    if pool == "gap":
        x = tf.keras.layers.GlobalAveragePooling1D()(x)
    else:  # 'last'
        # If the tensor is still 3D, retain the final time step
        if len(x.shape) == 3:
            x = tf.keras.layers.Lambda(lambda t: t[:, -1, :])(x)

    # Regression head
    if dense_dropout and dense_dropout > 0:
        x = tf.keras.layers.Dropout(dense_dropout)(x)
    x = tf.keras.layers.Dense(dense_units, activation="relu", kernel_regularizer=reg)(x)
    out = tf.keras.layers.Dense(1, kernel_regularizer=reg)(x)

    model = tf.keras.Model(inp, out)
    opt = (tf.keras.optimizers.AdamW(learning_rate=lr, weight_decay=l2reg, clipnorm=1.0)
           if (use_adamw and hasattr(tf.keras.optimizers, "AdamW"))
           else tf.keras.optimizers.Adam(learning_rate=lr, clipnorm=1.0))
    # The loss is supplied externally because it depends on the fold-specific sc_y
    model.compile(optimizer=opt, loss="mae")
    return model

# =========================================
# 8) HYPERPARAMETER OPTIMIZATION
# =========================================
import gc
import optuna
from optuna.trial import TrialState

N_TRIALS_LOW = 5       # 25 for definitive runs
N_TRIALS_HIGH = 1      # 5 for definitive runs
N_FOLDS = 3
windows = [24, 32, 40, 48, 56, 72]

# External boundary: TEST remains completely outside HPO
cache = {
    W: build_dataset_no_leak(df, W, p_train=P_TRAIN, p_val=P_VAL)
    for W in windows
}
HPO_END_TIME = min(pd.Timestamp(cache[W]["t_val_end"]) for W in windows)
df_hpo = df.loc[df.index <= HPO_END_TIME].copy()

future_times = df.index[df.index > HPO_END_TIME]

print("\n=========================================")
print("EXTERNAL HPO SEPARATION")
print("=========================================")
print("Last timestamp available to Optuna:", HPO_END_TIME)
print("First timestamp reserved outside HPO:", future_times.min())
print("HPO timestamps:", len(df_hpo))
print("Timestamps after HPO:", len(future_times))


def get_hpo_folds(df_hpo_slice, window, n_folds):
    """Generate walk-forward folds exclusively inside the HPO period."""
    ctx = build_features(df_hpo_slice)
    _, _, idx = build_seq2one_feats(
        ctx["feat_raw"].values,
        ctx["df"]["delta_t"].values,
        window,
        ctx["df"]["seg"].values
    )
    idx = np.asarray(idx, dtype=np.int64)

    if idx.size == 0:
        raise ValueError(f"No valid windows generated for WINDOW={window}.")

    return make_folds(ctx["df"].index[idx].to_numpy(), n_folds=n_folds)


def objective(trial, budget="low"):
    print(f"\n>>> Trial {trial.number + 1} ({budget}) started")

    window_trial = trial.suggest_categorical("WINDOW", windows)
    use_gru = trial.suggest_categorical("use_gru", [False, True])
    use_lstm2 = trial.suggest_categorical("use_lstm2", [True, False])
    pool = trial.suggest_categorical("pool", ["gap", "last"])
    lstm1 = trial.suggest_categorical("lstm1", [24, 32, 48, 64])
    lstm2 = (
        trial.suggest_categorical("lstm2", [8, 16, 24, 32])
        if use_lstm2 else 0
    )
    sp_dropout = trial.suggest_float("sp_dropout", 0.20, 0.50, step=0.05)
    rec_dropout = trial.suggest_float("rec_dropout", 0.10, 0.50, step=0.05)
    dense_units = trial.suggest_categorical("dense_units", [8, 16, 32])
    dense_dropout = trial.suggest_float("dense_dropout", 0.20, 0.50, step=0.05)
    noise_std = trial.suggest_float("noise_std", 0.00, 0.10, step=0.01)
    lr = trial.suggest_categorical("lr", [1e-3, 5e-4, 3e-4, 2e-4, 1e-4])
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128, 256])
    use_adamw = trial.suggest_categorical("use_adamw", [True, False])

    epochs_trial = 30 if budget == "low" else 80
    n_folds_trial = 2 if budget == "low" else N_FOLDS
    folds = get_hpo_folds(df_hpo, window_trial, n_folds_trial)

    validation_rmse = []
    training_rmse = []
    parameter_counts = []

    for fold_i, (train_end, validation_end) in enumerate(folds, start=1):
        print(
            f"    Fold {fold_i}/{len(folds)} | "
            f"train <= {train_end} | validation <= {validation_end}"
        )

        (
            X_fold, y_fold, times_fold, train_mask, validation_mask,
            sc_y_fold, reference_target
        ) = build_dataset_with_cuts(
            df_hpo, window_trial, train_end, validation_end
        )

        if not train_mask.any() or not validation_mask.any():
            raise ValueError(f"Fold {fold_i} contains an empty split.")

        X_train_fold = X_fold[train_mask].astype("float32")
        y_train_fold = y_fold[train_mask].astype("float32")
        X_validation_fold = X_fold[validation_mask].astype("float32")
        y_validation_fold = y_fold[validation_mask].astype("float32")

        reference_train = reference_target[train_mask]
        reference_validation = reference_target[validation_mask]

        # Extreme-event weights from training only
        residual_train = np.sinh(
            sc_y_fold.inverse_transform(y_train_fold)[:, 0]
        )
        Hs_train_true = reference_train + residual_train
        p95 = np.percentile(Hs_train_true, 95)

        sample_weights = np.ones_like(Hs_train_true, dtype=np.float32)
        sample_weights[Hs_train_true >= p95] = float(OVERSAMPLE_WEIGHT)

        tf.keras.backend.clear_session()

        fold_model = build_lstm_model(
            n_feats=X_fold.shape[-1],
            WINDOW=window_trial,
            lstm1=lstm1,
            lstm2=lstm2,
            use_lstm2=use_lstm2,
            use_gru=use_gru,
            pool=pool,
            sp_dropout=sp_dropout,
            rec_dropout=rec_dropout,
            dense_units=dense_units,
            dense_dropout=dense_dropout,
            lr=lr,
            l2reg=0.0,
            noise_std=noise_std,
            use_adamw=use_adamw
        )

        fold_model.compile(
            optimizer=fold_model.optimizer,
            loss=mae_meters_residual_loss(sc_y_fold)
        )

        callbacks = [
            EarlyStopping(
                monitor="val_loss",
                patience=4,
                min_delta=1e-4,
                restore_best_weights=True
            ),
            ReduceLROnPlateau(
                monitor="val_loss",
                patience=2,
                factor=0.5,
                min_lr=1e-5
            )
        ]

        history_fold = fold_model.fit(
            X_train_fold,
            y_train_fold,
            validation_data=(X_validation_fold, y_validation_fold),
            sample_weight=sample_weights,
            epochs=epochs_trial,
            batch_size=batch_size,
            verbose=0,
            callbacks=callbacks
        )

        # Optuna reporting
        for epoch_i, val_loss in enumerate(history_fold.history["val_loss"]):
            step = (fold_i - 1) * epochs_trial + epoch_i
            trial.report(float(val_loss), step=step)

            if trial.should_prune():
                del fold_model
                tf.keras.backend.clear_session()
                gc.collect()
                raise optuna.exceptions.TrialPruned()

        # Validation predictions
        pred_validation_scaled = fold_model.predict(
            X_validation_fold, verbose=0
        ).ravel()
        pred_validation_residual = np.sinh(
            sc_y_fold.inverse_transform(
                pred_validation_scaled.reshape(-1, 1)
            ).ravel()
        )
        Hs_validation_pred = reference_validation + pred_validation_residual

        true_validation_residual = np.sinh(
            sc_y_fold.inverse_transform(y_validation_fold).ravel()
        )
        Hs_validation_true = reference_validation + true_validation_residual

        rmse_validation = np.sqrt(
            mean_squared_error(Hs_validation_true, Hs_validation_pred)
        )
        validation_rmse.append(float(rmse_validation))

        # Training predictions
        pred_train_scaled = fold_model.predict(
            X_train_fold, verbose=0
        ).ravel()
        pred_train_residual = np.sinh(
            sc_y_fold.inverse_transform(
                pred_train_scaled.reshape(-1, 1)
            ).ravel()
        )
        Hs_train_pred = reference_train + pred_train_residual

        true_train_residual = np.sinh(
            sc_y_fold.inverse_transform(y_train_fold).ravel()
        )
        Hs_train_true = reference_train + true_train_residual

        rmse_train = np.sqrt(
            mean_squared_error(Hs_train_true, Hs_train_pred)
        )
        training_rmse.append(float(rmse_train))

        parameter_counts.append(
            int(np.sum([
                np.prod(variable.shape)
                for variable in fold_model.trainable_weights
            ]))
        )

        print(
            f"        RMSE train={rmse_train:.4f} m | "
            f"RMSE validation={rmse_validation:.4f} m"
        )

        del fold_model
        tf.keras.backend.clear_session()
        gc.collect()

    mean_val_rmse = float(np.mean(validation_rmse))
    mean_train_rmse = float(np.mean(training_rmse))
    val_rmse_std = float(np.std(validation_rmse))
    ratio = mean_val_rmse / max(mean_train_rmse, 1e-6)
    gap = max(0.0, mean_val_rmse - mean_train_rmse)
    n_parameters = int(np.mean(parameter_counts))

    objective_value = (
        mean_val_rmse
        + 0.75 * gap
        + 0.25 * max(0.0, ratio - 1.0)
        + 1e-7 * n_parameters
        + 0.25 * val_rmse_std
    )

    trial.set_user_attr("budget", budget)
    trial.set_user_attr("mean_validation_rmse", mean_val_rmse)
    trial.set_user_attr("mean_training_rmse", mean_train_rmse)
    trial.set_user_attr("validation_rmse_std", val_rmse_std)
    trial.set_user_attr("generalization_gap", gap)
    trial.set_user_attr("parameter_count", n_parameters)

    print(
        f">>> Trial {trial.number + 1} completed | "
        f"validation={mean_val_rmse:.3f} m | "
        f"train={mean_train_rmse:.3f} m | "
        f"gap={gap:.3f} m | objective={objective_value:.3f}"
    )

    return objective_value


# =========================================
# 8.4) LOW-BUDGET PHASE
# =========================================
study_low = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(
        seed=42, multivariate=True, group=True
    ),
    pruner=optuna.pruners.HyperbandPruner(
        min_resource=10, max_resource=30, reduction_factor=3
    ),
    study_name="GRU_LSTM_low_budget"
)

study_low.optimize(
    lambda trial: objective(trial, budget="low"),
    n_trials=N_TRIALS_LOW,
    gc_after_trial=True
)

completed_low = sorted(
    [
        trial for trial in study_low.trials
        if trial.state == TrialState.COMPLETE and trial.value is not None
    ],
    key=lambda trial: trial.value
)

if not completed_low:
    raise RuntimeError("No low-budget trial completed successfully.")

top_low_trials = completed_low[:min(5, len(completed_low))]

print("\nBEST LOW-BUDGET CANDIDATES")
for rank, trial in enumerate(top_low_trials, start=1):
    print(
        f"{rank}. Trial {trial.number} | "
        f"objective={trial.value:.4f} | params={trial.params}"
    )


# =========================================
# 8.5) HIGH-BUDGET PHASE
# =========================================
study_high = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(
        seed=42, multivariate=True, group=True
    ),
    pruner=optuna.pruners.HyperbandPruner(
        min_resource=10, max_resource=80, reduction_factor=3
    ),
    study_name="GRU_LSTM_high_budget"
)

for candidate in top_low_trials:
    study_high.enqueue_trial(candidate.params)

study_high.optimize(
    lambda trial: objective(trial, budget="high"),
    n_trials=max(N_TRIALS_HIGH, len(top_low_trials)),
    gc_after_trial=True
)

completed_high = [
    trial for trial in study_high.trials
    if trial.state == TrialState.COMPLETE and trial.value is not None
]

if not completed_high:
    raise RuntimeError("No high-budget trial completed successfully.")

best_trial = min(completed_high, key=lambda trial: trial.value)
best = best_trial.params

print("\n=========================================")
print("BEST HIGH-BUDGET CONFIGURATION")
print("=========================================")
print("Best objective:", best_trial.value)
print("Best hyperparameters:", best)
print("Mean validation RMSE:", best_trial.user_attrs["mean_validation_rmse"])
print("Mean training RMSE:", best_trial.user_attrs["mean_training_rmse"])
print("Generalization gap:", best_trial.user_attrs["generalization_gap"])


# =========================================
# 9) FINAL TRAINING AND EXTERNAL TEST
# =========================================
WINDOW = best.get("WINDOW", WINDOW)
BATCH = best.get("batch_size", BATCH)

data = build_dataset_no_leak(df, WINDOW, P_TRAIN, P_VAL)

X, y, times = data["X"], data["y"], data["times"]
train_mask = data["m_tr"]
validation_mask = data["m_val"]
test_mask = data["m_te"]
sc_y = data["sc_y"]
feat_scaled = data["feat_scaled"]

X_train = X[train_mask].astype("float32")
y_train = y[train_mask].astype("float32")
X_validation = X[validation_mask].astype("float32")
y_validation = y[validation_mask].astype("float32")
X_test = X[test_mask].astype("float32")
y_test = y[test_mask].astype("float32")

print("\n=========================================")
print("FINAL TEMPORAL SPLIT")
print("=========================================")
print("Train:", X_train.shape)
print("Validation:", X_validation.shape)
print("Test:", X_test.shape)
print("End of training:", data["t_train_end"])
print("End of validation:", data["t_val_end"])
print("Start of test:", times[test_mask].min())

# Training-only extreme weights
residual_train = np.sinh(sc_y.inverse_transform(y_train)[:, 0])
Hs_train = data["sim_t"][train_mask] + residual_train
p95 = np.percentile(Hs_train, 95)

training_weights = np.ones_like(Hs_train, dtype=np.float32)
training_weights[Hs_train >= p95] = float(OVERSAMPLE_WEIGHT)

tf.keras.backend.clear_session()

model = build_lstm_model(
    n_feats=X.shape[-1],
    WINDOW=WINDOW,
    lstm1=best.get("lstm1", 64),
    lstm2=best.get("lstm2", 32),
    use_lstm2=best.get("use_lstm2", True),
    use_gru=best.get("use_gru", False),
    pool=best.get("pool", "last"),
    sp_dropout=best.get("sp_dropout", 0.30),
    rec_dropout=best.get("rec_dropout", 0.10),
    dense_units=best.get("dense_units", 16),
    dense_dropout=best.get("dense_dropout", 0.30),
    lr=best.get("lr", 3e-4),
    l2reg=0.0,
    noise_std=best.get("noise_std", 0.0),
    use_adamw=best.get("use_adamw", False)
)

model.compile(
    optimizer=model.optimizer,
    loss=mae_meters_residual_loss(sc_y)
)

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_validation, y_validation),
    sample_weight=training_weights,
    epochs=EPOCHS,
    batch_size=BATCH,
    verbose=0,
    callbacks=[
        EarlyStopping(
            monitor="val_loss",
            patience=8,
            min_delta=1e-4,
            restore_best_weights=True
        ),
        ReduceLROnPlateau(
            monitor="val_loss",
            patience=4,
            factor=0.5,
            min_lr=1e-6
        )
    ]
)

print("Best validation loss:", float(np.min(history.history["val_loss"])))


# =========================================
# 9.1) EXTERNAL TEST
# =========================================
test_pred_scaled = model.predict(X_test, verbose=0).ravel()
test_pred_residual = np.sinh(
    sc_y.inverse_transform(test_pred_scaled.reshape(-1, 1)).ravel()
)
Hs_test_pred = data["sim_t"][test_mask] + test_pred_residual

test_true_residual = np.sinh(
    sc_y.inverse_transform(y_test).ravel()
)
Hs_test_true = data["sim_t"][test_mask] + test_true_residual

test_mae = mean_absolute_error(Hs_test_true, Hs_test_pred)
test_rmse = np.sqrt(mean_squared_error(Hs_test_true, Hs_test_pred))
test_r2 = r2_score(Hs_test_true, Hs_test_pred)

print("\n=========================================")
print("EXTERNAL TEST RESULTS")
print("=========================================")
print(
    f"MAE={test_mae:.3f} m | "
    f"RMSE={test_rmse:.3f} m | "
    f"R²={test_r2:.3f}"
)


# =========================================
# 10) TARGETED GAP RECONSTRUCTION
# =========================================


reconstructed = pd.Series(
    np.nan,
    index=df.index,
    name="Hs_reconstructed"
)

observed_mask = df["Hs_observed"].notna()
reconstructed.loc[observed_mask] = df.loc[
    observed_mask, "Hs_observed"
].values

missing_indices = np.where(df["Hs_observed"].isna())[0]
scaled_features = feat_scaled.values
segments = df["seg"].values

valid_gap_indices = [
    k for k in missing_indices
    if (
        k >= WINDOW - 1
        and segments[k] == segments[k - WINDOW + 1]
        and not np.isnan(
            scaled_features[k - WINDOW + 1:k + 1]
        ).any()
    )
]

print("\n=========================================")
print("GAP RECONSTRUCTION")
print("=========================================")
print("Total missing timestamps:", len(missing_indices))
print("Valid reconstruction windows:", len(valid_gap_indices))


if valid_gap_indices:
    X_gap = np.stack(
        [
            scaled_features[k - WINDOW + 1:k + 1]
            for k in valid_gap_indices
        ],
        axis=0
    ).astype("float32")

    gap_pred_scaled = model.predict(X_gap, verbose=0).ravel()
    gap_pred_residual = np.sinh(
        sc_y.inverse_transform(
            gap_pred_scaled.reshape(-1, 1)
        ).ravel()
    )

reference_at_gap = df["Hs_reference"].values[valid_gap_indices]

Hs_gap_pred = reference_at_gap + gap_pred_residual

reconstructed.iloc[valid_gap_indices] = Hs_gap_pred

final_series = reconstructed.copy()
final_series.name = "Hs_reconstructed_final"

if observed_mask.any():
    maximum_observation_difference = np.nanmax(
        np.abs(
            final_series.loc[observed_mask].values
            - df.loc[observed_mask, "Hs_observed"].values
        )
    )
else:
    maximum_observation_difference = np.nan

print(
    "Maximum difference in original observations:",
    maximum_observation_difference
)
print(
    "Reconstructed timestamps:",
    int(final_series.loc[df["Hs_observed"].isna()].notna().sum())
)



# =========================================
# 11) PLOTS
# =========================================
plt.figure(figsize=(13, 4))
plt.plot(
    df.index,
    df["Hs_reference"],
    label="Reference series",
    alpha=0.7,
    linewidth=1
)
plt.plot(
    final_series.index,
    final_series,
    label="GRU/LSTM reconstructed series",
    linewidth=1.5
)
plt.title("Significant wave height reconstruction")
plt.xlabel("Time")
plt.ylabel(r"$H_s$ (m)")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

plt.figure(figsize=(13, 4))
plt.plot(
    final_series.index,
    final_series,
    label="Reconstructed series",
    linewidth=1.5,
    color="black"
)
plt.plot(
    df.index,
    df["Hs_observed"],
    label="Observed series",
    linewidth=1,
    alpha=0.7,
    color="red"
)
plt.title("Observed and reconstructed significant wave height")
plt.xlabel("Time")
plt.ylabel(r"$H_s$ (m)")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# =========================================
# SAVE FINAL CNN-GRU TIME SERIES
# =========================================

from pathlib import Path

OUTPUT_DIR = Path("model_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

output_file = OUTPUT_DIR / "M1_reconstructed_CNN_GRU.csv"

# Final value:
# - observed value where measurements exist
# - DL reconstruction where observations are missing
output_df = pd.DataFrame(
    {
        "Hs": final_series,
        "source": np.where(
            df["Hs_observed"].notna(),
            "observed",
            np.where(
                final_series.notna(),
                "reconstructed",
                "unavailable"
            )
        )
    },
    index=final_series.index
)

output_df.index.name = "datetime"

output_df.to_csv(
    output_file,
    index=True
)

print(f"CSV saved successfully: {output_file}")
print("Rows saved:", len(output_df))
print("\nSource counts:")
print(output_df["source"].value_counts())


In [ ]:
# ===== Plot and export to SVG =====
# ===== Gappy observations vs final CNN-GRU series =====

import matplotlib.pyplot as plt
from pathlib import Path

save_dir = OUTPUT_DIR
save_dir.mkdir(parents=True, exist_ok=True)

# Retrieve the chronological split boundaries
# from the final leakage-free dataset
t_train_end = data["t_train_end"]
t_val_end   = data["t_val_end"]

# Masks based on the chronological temporal splits
mask_tr  = (df.index <= t_train_end)
mask_val = (df.index > t_train_end) & (df.index <= t_val_end)
mask_te  = (df.index > t_val_end)

hs_tr  = df["Hs_observed"].where(mask_tr)
hs_val = df["Hs_observed"].where(mask_val)
hs_te  = df["Hs_observed"].where(mask_te)
# --- Figure ---
plt.figure(figsize=(13, 4))

plt.plot(
    final_series.index,
    final_series,
    label="Final series (observed + reconstructed gaps)",
    linewidth=2,
    color="tab:blue"
)

# Plot observed values in each chronological subset
if hs_tr.notna().any():
    plt.plot(
        df.index,
        hs_tr,
        label="Train observations",
        alpha=0.7,
        color="0.5"
    )

if hs_val.notna().any():
    plt.plot(
        df.index,
        hs_val,
        label="Validation observations",
        alpha=0.8,
        color="tab:orange"
    )

if hs_te.notna().any():
    plt.plot(
        df.index,
        hs_te,
        label="Test observations",
        alpha=0.8,
        color="tab:green"
    )

# Vertical lines separating the chronological splits
plt.axvline(
    t_train_end,
    color="k",
    ls="--",
    lw=1,
    alpha=0.6
)

plt.axvline(
    t_val_end,
    color="k",
    ls="--",
    lw=1,
    alpha=0.6
)

# --- Styling and labels ---
plt.title(
    "Gappy observations and final reconstruction (CNN-GRU)"
)
plt.xlabel("Time")
plt.ylabel("Significant wave height (m)")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()

# --- Save as SVG ---
svg_path = (
    save_dir
    / "Gaps_vs_final_reconstruction_CNN_GRU.svg"
)

plt.savefig(
    svg_path,
    format="svg",
    bbox_inches="tight",
    dpi=300
)

plt.show()

print(f"Figure saved to: {svg_path}")

In [ ]:
# =========================================
# TRAIN / VALIDATION / TEST SCATTER PLOTS
# =========================================

def inverse_residual(normalized_values, scaler):
    """Convert normalized asinh residuals back to metres."""
    values_asinh = scaler.inverse_transform(
        np.asarray(normalized_values).reshape(-1, 1)
    ).ravel()
    return np.sinh(values_asinh)


def calculate_metrics(y_true, y_pred):
    """Return MAE, RMSE and R²."""
    valid = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = np.asarray(y_true)[valid]
    y_pred = np.asarray(y_pred)[valid]

    return (
        mean_absolute_error(y_true, y_pred),
        np.sqrt(mean_squared_error(y_true, y_pred)),
        r2_score(y_true, y_pred)
    )


# Reference values aligned with each subset
reference_train = data["sim_t"][train_mask]
reference_validation = data["sim_t"][validation_mask]
reference_test = data["sim_t"][test_mask]

# Model predictions in normalized residual space
pred_train_scaled = model.predict(X_train, verbose=0).ravel()
pred_validation_scaled = model.predict(X_validation, verbose=0).ravel()
pred_test_scaled = model.predict(X_test, verbose=0).ravel()

# True residuals in metres
residual_train_true = inverse_residual(y_train, sc_y)
residual_validation_true = inverse_residual(y_validation, sc_y)
residual_test_true = inverse_residual(y_test, sc_y)

# Predicted residuals in metres
residual_train_pred = inverse_residual(pred_train_scaled, sc_y)
residual_validation_pred = inverse_residual(
    pred_validation_scaled, sc_y
)
residual_test_pred = inverse_residual(pred_test_scaled, sc_y)

# Reconstruct total Hs = reference + residual
Hs_train_true = reference_train + residual_train_true
Hs_validation_true = reference_validation + residual_validation_true
Hs_test_true = reference_test + residual_test_true

Hs_train_pred = reference_train + residual_train_pred
Hs_validation_pred = reference_validation + residual_validation_pred
Hs_test_pred = reference_test + residual_test_pred

# Metrics
mae_train, rmse_train, r2_train = calculate_metrics(
    Hs_train_true, Hs_train_pred
)
mae_validation, rmse_validation, r2_validation = calculate_metrics(
    Hs_validation_true, Hs_validation_pred
)
mae_test, rmse_test, r2_test = calculate_metrics(
    Hs_test_true, Hs_test_pred
)

print(
    f"[TRAIN] MAE={mae_train:.3f} m | "
    f"RMSE={rmse_train:.3f} m | R²={r2_train:.3f}"
)
print(
    f"[VALIDATION] MAE={mae_validation:.3f} m | "
    f"RMSE={rmse_validation:.3f} m | R²={r2_validation:.3f}"
)
print(
    f"[TEST] MAE={mae_test:.3f} m | "
    f"RMSE={rmse_test:.3f} m | R²={r2_test:.3f}"
)

# Data for plots
splits = [
    (
        "TRAIN",
        Hs_train_true,
        Hs_train_pred,
        mae_train,
        rmse_train,
        r2_train
    ),
    (
        "VALIDATION",
        Hs_validation_true,
        Hs_validation_pred,
        mae_validation,
        rmse_validation,
        r2_validation
    ),
    (
        "TEST",
        Hs_test_true,
        Hs_test_pred,
        mae_test,
        rmse_test,
        r2_test
    )
]

split_colors = {
    "TRAIN": "gray",
    "VALIDATION": "tab:orange",
    "TEST": "tab:green"
}

# Common axis limits
all_values = np.concatenate([
    np.asarray(split[1]).ravel()
    for split in splits
] + [
    np.asarray(split[2]).ravel()
    for split in splits
])

all_values = all_values[np.isfinite(all_values)]

if all_values.size == 0:
    raise ValueError("No finite values are available for the scatter plots.")

minimum_value = float(np.min(all_values))
maximum_value = float(np.max(all_values))

padding = 0.05 * max(maximum_value - minimum_value, 1.0)
lower_limit = max(0.0, minimum_value - padding)
upper_limit = maximum_value + padding

# Horizontal scatter plots
fig, axes = plt.subplots(
    1, 3,
    figsize=(18, 6),
    sharex=True,
    sharey=True
)

for ax, (
    title,
    y_true,
    y_pred,
    mae,
    rmse,
    r2
) in zip(axes, splits):

    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()

    valid = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true_valid = y_true[valid]
    y_pred_valid = y_pred[valid]

    ax.scatter(
        y_true_valid,
        y_pred_valid,
        s=10,
        alpha=0.55,
        edgecolors="none",
        color=split_colors[title],
        label=title
    )

    ax.plot(
        [lower_limit, upper_limit],
        [lower_limit, upper_limit],
        linestyle="--",
        linewidth=1.5,
        color="black",
        label="1:1 line"
    )

    ax.set_title(title)
    ax.set_xlabel(r"Observed $H_s$ (m)")
    ax.set_xlim(lower_limit, upper_limit)
    ax.set_ylim(lower_limit, upper_limit)
    ax.set_aspect("equal", adjustable="box")
    ax.grid(True, linestyle="--", alpha=0.3)
    ax.legend(loc="lower right")

    ax.annotate(
        f"MAE = {mae:.2f} m\n"
        f"RMSE = {rmse:.2f} m\n"
        f"R² = {r2:.2f}",
        xy=(0.03, 0.97),
        xycoords="axes fraction",
        va="top",
        ha="left",
        bbox={
            "boxstyle": "round",
            "facecolor": "white",
            "edgecolor": "gray",
            "alpha": 0.8
        }
    )

axes[0].set_ylabel(r"Predicted $H_s$ (m)")

plt.suptitle(
    "Observed versus predicted significant wave height",
    fontsize=14,
    y=1.02
)

plt.tight_layout()
plt.show()

In [ ]:
# =========================================
# ADDITIONAL TEST METRICS
# =========================================
Hs_test_true = np.asarray(Hs_test_true).ravel()
Hs_test_pred = np.asarray(Hs_test_pred).ravel()

valid_test = np.isfinite(Hs_test_true) & np.isfinite(Hs_test_pred)

Hs_test_true_valid = Hs_test_true[valid_test]
Hs_test_pred_valid = Hs_test_pred[valid_test]

test_error = Hs_test_pred_valid - Hs_test_true_valid

test_mae = mean_absolute_error(
    Hs_test_true_valid,
    Hs_test_pred_valid
)

test_rmse = np.sqrt(
    mean_squared_error(
        Hs_test_true_valid,
        Hs_test_pred_valid
    )
)

test_r2 = r2_score(
    Hs_test_true_valid,
    Hs_test_pred_valid
)

if len(Hs_test_true_valid) > 1:
    test_r = float(
        np.corrcoef(
            Hs_test_true_valid,
            Hs_test_pred_valid
        )[0, 1]
    )
else:
    test_r = np.nan

test_bias = float(
    np.mean(test_error)
)

# Robust MAPE
denominator = np.maximum(
    np.abs(Hs_test_true_valid),
    1e-6
)

test_mape = float(
    np.mean(
        np.abs(test_error) / denominator
    ) * 100.0
)

print(
    f"[TEST] MAE={test_mae:.3f} m | "
    f"RMSE={test_rmse:.3f} m | "
    f"R²={test_r2:.3f} | "
    f"r={test_r:.3f} | "
    f"Bias={test_bias:.3f} m | "
    f"MAPE={test_mape:.1f}%"
)


# =========================================
# TEST TIME SERIES
# =========================================
test_times = pd.DatetimeIndex(times[test_mask])

# Apply the same finite mask used for the metrics
test_times_valid = test_times[valid_test]

plt.figure(figsize=(14, 4))

plt.plot(
    test_times_valid,
    Hs_test_true_valid,
    label="Observed TEST values",
    color="green",
    linewidth=1.8
)

plt.plot(
    test_times_valid,
    Hs_test_pred_valid,
    label="Model prediction",
    linewidth=1.6,
    alpha=0.9
)

plt.title(
    "TEST time series: observed versus predicted significant wave height"
)

plt.xlabel("Time")
plt.ylabel(r"Significant wave height $H_s$ (m)")
plt.grid(True, linestyle="--", alpha=0.35)
plt.legend()

metrics_text = (
    f"MAE:  {test_mae:.3f} m\n"
    f"RMSE: {test_rmse:.3f} m\n"
    f"R²:   {test_r2:.3f}\n"
    f"r:    {test_r:.3f}\n"
    f"Bias: {test_bias:.3f} m\n"
    f"MAPE: {test_mape:.1f}%"
)

ax = plt.gca()

ax.text(
    0.01,
    0.98,
    metrics_text,
    transform=ax.transAxes,
    va="top",
    ha="left",
    bbox={
        "boxstyle": "round",
        "facecolor": "white",
        "edgecolor": "gray",
        "alpha": 0.8
    }
)

plt.tight_layout()
plt.show()

## Temporal Convolutional Network model

This section implements the Temporal Convolutional Network (TCN) used to
reconstruct missing significant wave-height observations from the public
offshore buoy record.

The model uses causal dilated convolutions to capture temporal dependencies
across multiple time scales while preserving the chronological structure
of the input sequences.

The same preprocessing, feature set, chronological data splitting,
rolling-origin validation strategy, residual formulation, and evaluation
metrics used for the recurrent model are maintained to ensure a consistent
comparison between architectures.

In [ ]:
# -*- coding: utf-8 -*-
# Gap reconstruction in Hs_observed using TCN with improvements for extreme conditions
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras import regularizers
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy('float32')  
tf.keras.backend.set_floatx('float32')

# =========================================
# 0) Configuration
# =========================================
np.random.seed(42)
tf.random.set_seed(42)

# Window in 3-hour steps
WINDOW = 56
BATCH  = 64
EPOCHS = 200
P_TRAIN, P_VAL = 0.70, 0.10   


OVERSAMPLE_WEIGHT = 4.5  # weight assigned to Hs >= training P95
HUBER_DELTA = 1        # in normalized space (StandardScaler)

# ==== Residual loss in METRES using the fold-specific sc_y scaler ====
def mae_meters_residual_loss(sc_y):
    # y_true and y_pred are in normalized asinh(delta) space
    def loss(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.cast(y_pred, tf.float32)
        # inverse transformation of the fold-specific StandardScaler
        y_true_t = y_true * sc_y.scale_[0] + sc_y.mean_[0]
        y_pred_t = y_pred * sc_y.scale_[0] + sc_y.mean_[0]
        # convert from asinh(delta) to delta in metres
        y_true_m = tf.sinh(y_true_t)
        y_pred_m = tf.sinh(y_pred_t)
        return tf.reduce_mean(tf.abs(y_true_m - y_pred_m))
    return loss

# =========================================
# 1) Temporal selection (adjust the range if required)
# =========================================
# 1) Temporal selection without resampling
df = df_all.loc['1990-01-01':'2026-01-01', ['Hs_observed', 'Hs_reference']].copy().sort_index()

df = df.resample('3h').mean()

# --- Segments: break at time-step changes or when the reference is missing ---
deltas = df.index.to_series().diff()
freq   = deltas.mode().iloc[0]
tol    = pd.Timedelta('5min')
same_step = deltas.between(freq - tol, freq + tol)

df['seg'] = (~same_step | df['Hs_reference'].isna()).cumsum().fillna(0).astype(int)

# =========================================
# 2) Chronological temporal split by percentages
# =========================================
def temporal_split_percent(times_index, p_train=0.70, p_val=0.10):
    times_index = pd.DatetimeIndex(times_index)

    if len(times_index) == 0:
        raise ValueError("times_index is empty.")
    if not (0 < p_train < 1 and 0 < p_val < 1 and p_train + p_val < 1):
        raise ValueError("Invalid percentages.")

    order = np.argsort(times_index.values)
    times_sorted = times_index.values[order]
    n = len(times_sorted)

    i_train_end = int(np.floor(p_train * (n - 1)))
    i_val_end   = int(np.floor((p_train + p_val) * (n - 1)))

    t_train_end = pd.to_datetime(times_sorted[i_train_end])
    t_val_end   = pd.to_datetime(times_sorted[i_val_end])

    m_tr  = (times_index <= t_train_end)
    m_val = (times_index >  t_train_end) & (times_index <= t_val_end)
    m_te  = (times_index >  t_val_end)

    if not (m_tr.any() and m_val.any() and m_te.any()):
        raise ValueError(
            f"Empty split. Counts -> train:{m_tr.sum()}, val:{m_val.sum()}, test:{m_te.sum()}"
        )
    return m_tr, m_val, m_te, t_train_end, t_val_end

# =========================================
# 3) Reference features and transformed target
#    - Target: z = asinh(Hs_observed)
#    - Features: reference plus rolling statistics (1, 3, and 7 days) without leakage (past values only)
# =========================================
# Target in transformed space (NaNs may be present)
df['delta']   = df['Hs_observed'] - df['Hs_reference']
df['delta_t'] = np.arcsinh(df['delta'])    # target in asinh space
# Base reference signal used for rolling features
sim_base = df['Hs_reference']


roll_steps = [8, 24, 56]
feat = pd.DataFrame(index=df.index)
feat['sim'] = sim_base

for h in roll_steps:
    r = sim_base.rolling(h, min_periods=h)   # requires a complete window
    feat[f'sim_max_{h}']  = r.max()
    feat[f'sim_mean_{h}'] = r.mean()
    feat[f'sim_std_{h}']  = r.std()

# =========================================
# 4) First windowing pass to obtain timestamps and define the split
#    (without scaling; used only to obtain timestamps and split boundaries)
# =========================================
def build_seq2one_feats(arr_feats, observed_target, window, seg_ids):
    X, y, t_idx = [], [], []
    for i in range(window - 1, len(arr_feats)):
        # 1) Do not cross segments (prevents windows from spanning long gaps)
        if seg_ids[i] != seg_ids[i - window + 1]:
            continue
        xs = arr_feats[i - window + 1:i + 1, :]

        # 2) Discard windows containing any NaN in the features or target
        if np.isnan(xs).any():
            continue
        if np.isnan(observed_target[i]):
            continue

        X.append(xs)
        y.append(observed_target[i])
        t_idx.append(i)
    return np.array(X), np.array(y).reshape(-1, 1), np.array(t_idx, dtype=np.int64)



# === 4) First pass to obtain timestamps and split boundaries using delta_t ===
X0, y0_dummy, t_idx0 = build_seq2one_feats(
    feat.values,                       # raw unscaled features
    df['delta_t'].values,              # residual target in asinh space
    WINDOW,
    df['seg'].values
)
t_idx0  = np.asarray(t_idx0, dtype=np.int64)
if t_idx0.size == 0:
    raise ValueError("No valid windows were generated during the first pass.")

times0 = df.index[t_idx0]
m_tr0, m_val0, m_te0, t_train_end, t_val_end = temporal_split_percent(times0, P_TRAIN, P_VAL)


# =========================================
# 5) Leakage-free scaling fitted only on the actual training subset
#    - StandardScaler for X (all columns)
#    - StandardScaler for y (asinh residual)
# =========================================
mask_scalers = (df.index <= t_train_end)

sc_X = StandardScaler()
sc_y = StandardScaler()



sc_X = StandardScaler().fit(feat.loc[mask_scalers].values)
sc_y = StandardScaler().fit(df.loc[mask_scalers, ['delta_t']].dropna())

feat_scaled = pd.DataFrame(
    sc_X.transform(feat.values),
    index=feat.index, columns=feat.columns
)

df['delta_scaled'] = np.nan
ok_delta = df['delta_t'].notna()
df.loc[ok_delta, 'delta_scaled'] = sc_y.transform(df.loc[ok_delta, ['delta_t']])[:, 0]


# =========================================
# 6) Final windowing after scaling
# =========================================
X, y, t_idx = build_seq2one_feats(
    feat_scaled.values,
    df['delta_scaled'].values,
    WINDOW,
    df['seg'].values
)
t_idx = np.asarray(t_idx, dtype=np.int64)
if t_idx.size == 0:
    raise ValueError("No valid windows were generated after scaling.")

times = df.index[t_idx]
m_tr  = (times <= t_train_end)
m_val = (times >  t_train_end) & (times <= t_val_end)
m_te  = (times >  t_val_end)

X_tr, y_tr = X[m_tr],  y[m_tr]
X_val, y_val = X[m_val], y[m_val]
X_te,  y_te  = X[m_te],  y[m_te]

print("Shapes -> Train:", X_tr.shape, "Val:", X_val.shape, "Test:", X_te.shape)


def build_features(df):
    out = {}
    dfx = df.copy()

    # Residual target
    dfx['delta']   = dfx['Hs_observed'] - dfx['Hs_reference']
    dfx['delta_t'] = np.arcsinh(dfx['delta'])

    # Reference base without propagation across gaps
    sim_base = dfx['Hs_reference']

    feat = pd.DataFrame(index=dfx.index)
    feat['sim'] = sim_base

    # Rolling statistics requiring complete windows
    for h in [8, 24, 56]:
        r = sim_base.rolling(h, min_periods=h)
        feat[f'sim_max_{h}']  = r.max()
        feat[f'sim_mean_{h}'] = r.mean()
        feat[f'sim_std_{h}']  = r.std()

    # Lagged variables using past values only
    for k in [1, 3, 8, 24, 56]:
        feat[f'sim_lag_{k}'] = sim_base.shift(k)

    # Dynamics
    feat['sim_diff_1'] = sim_base.diff(1)
    feat['sim_diff_3'] = sim_base.diff(3)
    feat['sim_diff_8'] = sim_base.diff(8)
    feat['sim_ema_8']  = sim_base.ewm(span=8,  adjust=False).mean()
    feat['sim_ema_24'] = sim_base.ewm(span=24, adjust=False).mean()
    feat['sim_ema_56'] = sim_base.ewm(span=56, adjust=False).mean()



    # Seasonality
    feat['sin_doy']  = np.sin(2*np.pi*feat.index.dayofyear/365.25)
    feat['cos_doy']  = np.cos(2*np.pi*feat.index.dayofyear/365.25)
    feat['sin_hour'] = np.sin(2*np.pi*feat.index.hour/24.0)
    feat['cos_hour'] = np.cos(2*np.pi*feat.index.hour/24.0)
    feat['sin_mon']  = np.sin(2*np.pi*(feat.index.month-1)/12)
    feat['cos_mon']  = np.cos(2*np.pi*(feat.index.month-1)/12)

    out['df'] = dfx
    out['feat_raw'] = feat
    return out




# ========= Walk-forward CV =========
def make_folds(times, n_folds=3):
    """Generate n_folds with progressively increasing cutoffs over times."""
    n = len(times)
    # Start the first fold near the middle to ensure a sufficiently large training subset
    idx = np.linspace(int(0.55*n), n-1, n_folds+1, dtype=int)
    folds = []
    for i in range(n_folds):
        i_train_end = idx[i]
        i_val_end   = idx[i+1]
        t_train_end = times[i_train_end]
        t_val_end   = times[i_val_end]
        folds.append((pd.Timestamp(t_train_end), pd.Timestamp(t_val_end)))
    return folds

def build_dataset_with_cuts(df_slice, WINDOW, t_train_end, t_val_end):
    """Leakage-free fold for the residual delta = Hs_observed - Hs_reference in asinh space."""
    dfl = df_slice.copy()
    if 'seg' in df.columns:
            dfl['seg'] = df.loc[dfl.index, 'seg'].to_numpy()
    else:
            deltas = dfl.index.to_series().diff()
            freq   = deltas.mode().iloc[0]
            tol    = pd.Timedelta('5min')
            same_step = deltas.between(freq - tol, freq + tol)
            dfl['seg'] = (~same_step).cumsum().fillna(0).astype(int)

    # features
    ctx  = build_features(dfl)
    feat = ctx['feat_raw']

    # Residual target in asinh space, required before scaling y
    with np.errstate(invalid='ignore'):
        dfl['delta']   = dfl['Hs_observed'] - dfl['Hs_reference']
        dfl['delta_t'] = np.arcsinh(dfl['delta'])

    # Fit scalers only on the fold training subset
    mask_scal = (dfl.index <= t_train_end)          # <- define beforehand
    sc_X_f = StandardScaler().fit(feat.loc[mask_scal].values)
    sc_y_f = StandardScaler().fit(dfl.loc[mask_scal, ['delta_t']].dropna())

    feat_sc = pd.DataFrame(sc_X_f.transform(feat.values), index=feat.index, columns=feat.columns)

    dfl['delta_scaled'] = np.nan
    ok = dfl['delta_t'].notna()
    dfl.loc[ok, 'delta_scaled'] = sc_y_f.transform(dfl.loc[ok, ['delta_t']])[:, 0]

    # Windowing: X contains features and y contains delta_scaled
    Xf, yf, t_idx = build_seq2one_feats(
    feat_sc.values, dfl['delta_scaled'].values, WINDOW, dfl['seg'].values
    )
    t_idx = np.asarray(t_idx, dtype=np.int64)
    times_f = dfl.index[t_idx]

    sim_t   = dfl['Hs_reference'].values[t_idx]

    m_tr  = (times_f <= t_train_end)
    m_val = (times_f > t_train_end) & (times_f <= t_val_end)
    return Xf, yf, times_f, m_tr, m_val, sc_y_f, sim_t


def build_dataset_no_leak(df_slice, WINDOW, p_train=P_TRAIN, p_val=P_VAL):
    """Global leakage-free train/validation/test split for the residual."""
    ctx    = build_features(df_slice)
    df_loc = ctx['df']
    feat   = ctx['feat_raw']

    if 'seg' in df.columns:
        df_loc['seg'] = df.loc[df_loc.index, 'seg'].to_numpy()
    else:
        deltas = df_loc.index.to_series().diff()
        freq   = deltas.mode().iloc[0]
        tol    = pd.Timedelta('5min')
        same_step = deltas.between(freq - tol, freq + tol)
        df_loc['seg'] = (~same_step).cumsum().fillna(0).astype(int)


    feat   = ctx['feat_raw']

    # Residual target in asinh space used to define split boundaries
    df_loc['delta']   = df_loc['Hs_observed'] - df_loc['Hs_reference']
    df_loc['delta_t'] = np.arcsinh(df_loc['delta'])

    # First pass for split boundaries using delta_t
    X0, _, t_idx0 = build_seq2one_feats(feat.values, df_loc['delta_t'].values, WINDOW, df_loc['seg'].values)
    t_idx0 = np.asarray(t_idx0, dtype=np.int64)
    times0 = df_loc.index[t_idx0]
    m_tr0, m_val0, m_te0, t_train_end, t_val_end = temporal_split_percent(times0, p_train, p_val)

    # Scalers fitted on training data
    mask_scal = (df_loc.index <= t_train_end)
    sc_X = StandardScaler().fit(feat.loc[mask_scal].values)
    sc_y = StandardScaler().fit(df_loc.loc[mask_scal, ['delta_t']].dropna())

    feat_sc = pd.DataFrame(sc_X.transform(feat.values), index=feat.index, columns=feat.columns)

    df_loc['delta_scaled'] = np.nan
    ok = df_loc['delta_t'].notna()
    df_loc.loc[ok, 'delta_scaled'] = sc_y.transform(df_loc.loc[ok, ['delta_t']])[:, 0]

    X, y, t_idx = build_seq2one_feats(feat_sc.values, df_loc['delta_scaled'].values, WINDOW, df_loc['seg'].values)
    t_idx = np.asarray(t_idx, dtype=np.int64)
    times = df_loc.index[t_idx]
    sim_t = df_loc['Hs_reference'].values[t_idx]

    m_tr, m_val, m_te, t_train_end, t_val_end = temporal_split_percent(times, p_train, p_val)

    return {
        'X': X, 'y': y, 'times': times,
        'm_tr': m_tr, 'm_val': m_val, 'm_te': m_te,
        't_train_end': t_train_end, 't_val_end': t_val_end,
        'sc_X': sc_X, 'sc_y': sc_y,
        'feat_scaled': feat_sc,
        'df_loc': df_loc,
        'sim_t': sim_t
    }
def _tcn_res_block(x, n_filters, ksize, dilation, dropout=0.20, l2reg=0.0):
    reg = tf.keras.regularizers.l2(l2reg) if (l2reg and l2reg > 0) else None
    h = layers.Conv1D(n_filters, ksize, padding="causal",
                      dilation_rate=dilation, activation="relu",
                      kernel_regularizer=reg)(x)
    h = layers.SpatialDropout1D(dropout)(h)
    h = layers.Conv1D(n_filters, ksize, padding="causal",
                      dilation_rate=dilation, activation="relu",
                      kernel_regularizer=reg)(h)
    if x.shape[-1] != n_filters:
        x = layers.Conv1D(n_filters, 1, padding="same", kernel_regularizer=reg)(x)
    return layers.Add()([x, h])
def _tcn_receptive_field(ksize, n_blocks):
    # RF is approximately k * (2^B - 1), used to verify that it covers the window
    return int(ksize * (2**n_blocks - 1))

def build_tcn_model(n_feats, WINDOW, n_blocks=5, n_filters=48, ksize=5,
                    base_dilation=1, dropout=0.20, l2reg=0.0,
                    gap_pool=True, lr=3e-4, use_adamw=False):
    reg = tf.keras.regularizers.l2(l2reg) if l2reg else None
    inp = layers.Input(shape=(WINDOW, n_feats))
    x = layers.Conv1D(n_filters, 3, padding="causal",
                      activation="relu", kernel_regularizer=reg)(inp)

    dilation = base_dilation
    for _ in range(n_blocks):
        x = _tcn_res_block(x, n_filters, ksize, dilation, dropout, l2reg)
        dilation *= 2

    x = layers.GlobalAveragePooling1D()(x) if gap_pool else layers.Lambda(lambda t: t[:, -1, :])(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(16, activation="relu", kernel_regularizer=reg)(x)
    out = layers.Dense(1, kernel_regularizer=reg)(x)
    model = tf.keras.Model(inp, out)

    opt = tf.keras.optimizers.AdamW(lr, weight_decay=l2reg, clipnorm=1.0) if use_adamw else \
          tf.keras.optimizers.Adam(lr, clipnorm=1.0)
    model.compile(optimizer=opt, loss="mae")  # the loss in metres is assigned externally, as before
    return model


# =========================================
# 8) TCN HYPERPARAMETER OPTIMIZATION
# =========================================
import gc
import optuna
from optuna.trial import TrialState

N_TRIALS_LOW = 5       # increase for more runs
N_TRIALS_HIGH = 3      # increase for more runs
N_FOLDS = 3
windows = [24, 32, 40, 48, 56, 72]

# External boundary: the future TEST period remains outside HPO
cache = {
    W: build_dataset_no_leak(df, W, p_train=P_TRAIN, p_val=P_VAL)
    for W in windows
}
HPO_END_TIME = min(pd.Timestamp(cache[W]["t_val_end"]) for W in windows)
df_hpo = df.loc[df.index <= HPO_END_TIME].copy()
future_times = df.index[df.index > HPO_END_TIME]

print("\n=========================================")
print("EXTERNAL HPO SEPARATION — TCN")
print("=========================================")
print("Last timestamp available to Optuna:", HPO_END_TIME)
print("First timestamp reserved outside HPO:", future_times.min())
print("HPO timestamps:", len(df_hpo))
print("Timestamps after HPO:", len(future_times))


def get_hpo_folds(df_hpo_slice, window, n_folds):
    """Generate walk-forward folds exclusively within the HPO period."""
    ctx = build_features(df_hpo_slice)
    _, _, idx = build_seq2one_feats(
        ctx["feat_raw"].values,
        ctx["df"]["delta_t"].values,
        window,
        ctx["df"]["seg"].values
    )
    idx = np.asarray(idx, dtype=np.int64)

    if idx.size == 0:
        raise ValueError(f"No valid windows generated for WINDOW={window}.")

    return make_folds(ctx["df"].index[idx].to_numpy(), n_folds=n_folds)


def objective(trial, budget="low"):
    print(f"\n>>> TCN trial {trial.number + 1} ({budget}) started")

    window_trial = trial.suggest_categorical("WINDOW", windows)
    n_blocks = trial.suggest_int("tcn_blocks", 4, 6)
    n_filters = trial.suggest_categorical("tcn_filters", [32, 48, 64])
    kernel_size = trial.suggest_categorical("tcn_ksize", [3, 5, 7])
    dropout = trial.suggest_float("tcn_dropout", 0.15, 0.40, step=0.05)
    l2reg = trial.suggest_float("l2reg", 0.0, 1e-4, step=1e-5)
    learning_rate = trial.suggest_categorical(
        "lr", [1e-3, 5e-4, 3e-4, 2e-4, 1e-4]
    )
    batch_size = trial.suggest_categorical(
        "batch_size", [32, 64, 128, 256]
    )
    use_adamw = trial.suggest_categorical("use_adamw", [True, False])

    epochs_trial = 30 if budget == "low" else 80
    n_folds_trial = 2 if budget == "low" else N_FOLDS
    folds = get_hpo_folds(df_hpo, window_trial, n_folds_trial)

    validation_rmse = []
    training_rmse = []
    parameter_counts = []

    for fold_i, (train_end, validation_end) in enumerate(folds, start=1):
        print(
            f"    Fold {fold_i}/{len(folds)} | "
            f"train <= {train_end} | validation <= {validation_end}"
        )

        (
            X_fold, y_fold, times_fold, train_mask, validation_mask,
            sc_y_fold, reference_target
        ) = build_dataset_with_cuts(
            df_hpo, window_trial, train_end, validation_end
        )

        if not train_mask.any() or not validation_mask.any():
            raise ValueError(f"Fold {fold_i} contains an empty split.")

        X_train_fold = X_fold[train_mask].astype("float32")
        y_train_fold = y_fold[train_mask].astype("float32")
        X_validation_fold = X_fold[validation_mask].astype("float32")
        y_validation_fold = y_fold[validation_mask].astype("float32")

        reference_train = reference_target[train_mask]
        reference_validation = reference_target[validation_mask]

        # Extreme-event weights calculated from training only
        residual_train = np.sinh(
            sc_y_fold.inverse_transform(y_train_fold)[:, 0]
        )
        Hs_train_true = reference_train + residual_train
        p95 = np.percentile(Hs_train_true, 95)

        sample_weights = np.ones_like(Hs_train_true, dtype=np.float32)
        sample_weights[Hs_train_true >= p95] = float(OVERSAMPLE_WEIGHT)

        tf.keras.backend.clear_session()

        fold_model = build_tcn_model(
            n_feats=X_fold.shape[-1],
            WINDOW=window_trial,
            n_blocks=n_blocks,
            n_filters=n_filters,
            ksize=kernel_size,
            base_dilation=1,
            dropout=dropout,
            l2reg=l2reg,
            gap_pool=True,
            lr=learning_rate,
            use_adamw=use_adamw
        )

        fold_model.compile(
            optimizer=fold_model.optimizer,
            loss=mae_meters_residual_loss(sc_y_fold)
        )

        fold_history = fold_model.fit(
            X_train_fold,
            y_train_fold,
            validation_data=(X_validation_fold, y_validation_fold),
            sample_weight=sample_weights,
            epochs=epochs_trial,
            batch_size=batch_size,
            verbose=0,
            callbacks=[
                EarlyStopping(
                    monitor="val_loss",
                    patience=4,
                    min_delta=1e-4,
                    restore_best_weights=True
                ),
                ReduceLROnPlateau(
                    monitor="val_loss",
                    patience=2,
                    factor=0.5,
                    min_lr=1e-5
                )
            ]
        )

        for epoch_i, val_loss in enumerate(fold_history.history["val_loss"]):
            step = (fold_i - 1) * epochs_trial + epoch_i
            trial.report(float(val_loss), step=step)

            if trial.should_prune():
                del fold_model
                tf.keras.backend.clear_session()
                gc.collect()
                raise optuna.exceptions.TrialPruned()

        # Validation RMSE
        validation_pred_scaled = fold_model.predict(
            X_validation_fold, verbose=0
        ).ravel()
        validation_pred_residual = np.sinh(
            sc_y_fold.inverse_transform(
                validation_pred_scaled.reshape(-1, 1)
            ).ravel()
        )
        Hs_validation_pred = (
            reference_validation + validation_pred_residual
        )

        validation_true_residual = np.sinh(
            sc_y_fold.inverse_transform(y_validation_fold).ravel()
        )
        Hs_validation_true = (
            reference_validation + validation_true_residual
        )

        rmse_validation = np.sqrt(
            mean_squared_error(Hs_validation_true, Hs_validation_pred)
        )
        validation_rmse.append(float(rmse_validation))

        # Training RMSE
        train_pred_scaled = fold_model.predict(
            X_train_fold, verbose=0
        ).ravel()
        train_pred_residual = np.sinh(
            sc_y_fold.inverse_transform(
                train_pred_scaled.reshape(-1, 1)
            ).ravel()
        )
        Hs_train_pred = reference_train + train_pred_residual

        train_true_residual = np.sinh(
            sc_y_fold.inverse_transform(y_train_fold).ravel()
        )
        Hs_train_true = reference_train + train_true_residual

        rmse_train = np.sqrt(
            mean_squared_error(Hs_train_true, Hs_train_pred)
        )
        training_rmse.append(float(rmse_train))

        parameter_counts.append(
            int(np.sum([
                np.prod(variable.shape)
                for variable in fold_model.trainable_weights
            ]))
        )

        print(
            f"        RMSE train={rmse_train:.4f} m | "
            f"RMSE validation={rmse_validation:.4f} m"
        )

        del fold_model
        tf.keras.backend.clear_session()
        gc.collect()

    mean_val_rmse = float(np.mean(validation_rmse))
    mean_train_rmse = float(np.mean(training_rmse))
    val_rmse_std = float(np.std(validation_rmse))
    ratio = mean_val_rmse / max(mean_train_rmse, 1e-6)
    gap = max(0.0, mean_val_rmse - mean_train_rmse)
    n_parameters = int(np.mean(parameter_counts))

    objective_value = (
        mean_val_rmse
        + 0.75 * gap
        + 0.25 * max(0.0, ratio - 1.0)
        + 1e-7 * n_parameters
        + 0.25 * val_rmse_std
    )

    trial.set_user_attr("budget", budget)
    trial.set_user_attr("mean_validation_rmse", mean_val_rmse)
    trial.set_user_attr("mean_training_rmse", mean_train_rmse)
    trial.set_user_attr("validation_rmse_std", val_rmse_std)
    trial.set_user_attr("generalization_gap", gap)
    trial.set_user_attr("parameter_count", n_parameters)

    print(
        f">>> TCN trial {trial.number + 1} completed | "
        f"validation={mean_val_rmse:.3f} m | "
        f"train={mean_train_rmse:.3f} m | "
        f"gap={gap:.3f} m | objective={objective_value:.3f}"
    )

    return objective_value


# =========================================
# 8.1) LOW-BUDGET PHASE
# =========================================
study_low = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(
        seed=42, multivariate=True, group=True
    ),
    pruner=optuna.pruners.HyperbandPruner(
        min_resource=10, max_resource=30, reduction_factor=3
    ),
    study_name="TCN_low_budget"
)

study_low.optimize(
    lambda trial: objective(trial, budget="low"),
    n_trials=N_TRIALS_LOW,
    gc_after_trial=True
)

completed_low = sorted(
    [
        trial for trial in study_low.trials
        if trial.state == TrialState.COMPLETE and trial.value is not None
    ],
    key=lambda trial: trial.value
)

if not completed_low:
    raise RuntimeError("No low-budget TCN trial completed successfully.")

top_low_trials = completed_low[:min(5, len(completed_low))]

print("\nBEST LOW-BUDGET TCN CANDIDATES")
for rank, trial in enumerate(top_low_trials, start=1):
    print(
        f"{rank}. Trial {trial.number} | "
        f"objective={trial.value:.4f} | params={trial.params}"
    )


# =========================================
# 8.2) HIGH-BUDGET PHASE
# =========================================
study_high = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(
        seed=42, multivariate=True, group=True
    ),
    pruner=optuna.pruners.HyperbandPruner(
        min_resource=10, max_resource=80, reduction_factor=3
    ),
    study_name="TCN_high_budget"
)

for candidate in top_low_trials:
    study_high.enqueue_trial(candidate.params)

study_high.optimize(
    lambda trial: objective(trial, budget="high"),
    n_trials=max(N_TRIALS_HIGH, len(top_low_trials)),
    gc_after_trial=True
)

completed_high = [
    trial for trial in study_high.trials
    if trial.state == TrialState.COMPLETE and trial.value is not None
]

if not completed_high:
    raise RuntimeError("No high-budget TCN trial completed successfully.")

best_trial = min(completed_high, key=lambda trial: trial.value)
best = best_trial.params

print("\n=========================================")
print("BEST HIGH-BUDGET TCN CONFIGURATION")
print("=========================================")
print("Best objective:", best_trial.value)
print("Best hyperparameters:", best)
print("Mean validation RMSE:", best_trial.user_attrs["mean_validation_rmse"])
print("Mean training RMSE:", best_trial.user_attrs["mean_training_rmse"])
print("Generalization gap:", best_trial.user_attrs["generalization_gap"])


# =========================================
# 9) FINAL TCN TRAINING AND EXTERNAL TEST
# =========================================
WINDOW = best.get("WINDOW", WINDOW)
BATCH = best.get("batch_size", BATCH)

data = build_dataset_no_leak(df, WINDOW, P_TRAIN, P_VAL)

X, y, times = data["X"], data["y"], data["times"]
train_mask = data["m_tr"]
validation_mask = data["m_val"]
test_mask = data["m_te"]
sc_y = data["sc_y"]
feat_scaled = data["feat_scaled"]

X_train = X[train_mask].astype("float32")
y_train = y[train_mask].astype("float32")
X_validation = X[validation_mask].astype("float32")
y_validation = y[validation_mask].astype("float32")
X_test = X[test_mask].astype("float32")
y_test = y[test_mask].astype("float32")

print("\n=========================================")
print("FINAL TCN TEMPORAL SPLIT")
print("=========================================")
print("Train:", X_train.shape)
print("Validation:", X_validation.shape)
print("Test:", X_test.shape)
print("End of training:", data["t_train_end"])
print("End of validation:", data["t_val_end"])
print("Start of test:", times[test_mask].min())

# Extreme-event weights derived from training only
residual_train = np.sinh(sc_y.inverse_transform(y_train)[:, 0])
Hs_train = data["sim_t"][train_mask] + residual_train
p95 = np.percentile(Hs_train, 95)

training_weights = np.ones_like(Hs_train, dtype=np.float32)
training_weights[Hs_train >= p95] = float(OVERSAMPLE_WEIGHT)

tf.keras.backend.clear_session()

model = build_tcn_model(
    n_feats=X.shape[-1],
    WINDOW=WINDOW,
    n_blocks=best.get("tcn_blocks", 5),
    n_filters=best.get("tcn_filters", 48),
    ksize=best.get("tcn_ksize", 5),
    base_dilation=1,
    dropout=best.get("tcn_dropout", 0.25),
    l2reg=best.get("l2reg", 0.0),
    gap_pool=True,
    lr=best.get("lr", 3e-4),
    use_adamw=best.get("use_adamw", False)
)

model.compile(
    optimizer=model.optimizer,
    loss=mae_meters_residual_loss(sc_y)
)

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_validation, y_validation),
    sample_weight=training_weights,
    epochs=EPOCHS,
    batch_size=BATCH,
    verbose=0,
    callbacks=[
        EarlyStopping(
            monitor="val_loss",
            patience=8,
            min_delta=1e-4,
            restore_best_weights=True
        ),
        ReduceLROnPlateau(
            monitor="val_loss",
            patience=4,
            factor=0.5,
            min_lr=1e-6
        )
    ]
)

print("Best validation loss:", float(np.min(history.history["val_loss"])))


# =========================================
# 9.1) EXTERNAL TEST
# =========================================
test_pred_scaled = model.predict(X_test, verbose=0).ravel()
test_pred_residual = np.sinh(
    sc_y.inverse_transform(test_pred_scaled.reshape(-1, 1)).ravel()
)
Hs_test_pred = data["sim_t"][test_mask] + test_pred_residual

test_true_residual = np.sinh(
    sc_y.inverse_transform(y_test).ravel()
)
Hs_test_true = data["sim_t"][test_mask] + test_true_residual

test_mae = mean_absolute_error(Hs_test_true, Hs_test_pred)
test_rmse = np.sqrt(mean_squared_error(Hs_test_true, Hs_test_pred))
test_r2 = r2_score(Hs_test_true, Hs_test_pred)

print("\n=========================================")
print("EXTERNAL TCN TEST RESULTS")
print("=========================================")
print(
    f"MAE={test_mae:.3f} m | "
    f"RMSE={test_rmse:.3f} m | "
    f"R²={test_r2:.3f}"
)


# =========================================
# 10) TARGETED GAP RECONSTRUCTION
# =========================================
HS_MIN, HS_MAX = 0.0, 20.0

reconstructed = pd.Series(
    np.nan, index=df.index, name="Hs_reconstructed"
)

# Preserve original observations exactly
observed_mask = df["Hs_observed"].notna()
reconstructed.loc[observed_mask] = df.loc[
    observed_mask, "Hs_observed"
].values

missing_indices = np.where(df["Hs_observed"].isna())[0]
scaled_features = feat_scaled.values
segments = df["seg"].values

valid_gap_indices = [
    k for k in missing_indices
    if (
        k >= WINDOW - 1
        and segments[k] == segments[k - WINDOW + 1]
        and not np.isnan(
            scaled_features[k - WINDOW + 1:k + 1]
        ).any()
    )
]

print("\n=========================================")
print("TCN GAP RECONSTRUCTION")
print("=========================================")
print("Total missing timestamps:", len(missing_indices))
print("Valid reconstruction windows:", len(valid_gap_indices))


if valid_gap_indices:
    X_gap = np.stack(
        [
            scaled_features[k - WINDOW + 1:k + 1]
            for k in valid_gap_indices
        ],
        axis=0
    ).astype("float32")

    gap_pred_scaled = model.predict(X_gap, verbose=0).ravel()
    gap_pred_residual = np.sinh(
        sc_y.inverse_transform(
            gap_pred_scaled.reshape(-1, 1)
        ).ravel()
    )

    reference_at_gap = df["Hs_reference"].values[valid_gap_indices]
    Hs_gap_pred = np.clip(
        reference_at_gap + gap_pred_residual,
        HS_MIN,
        HS_MAX
    )

    reconstructed.iloc[valid_gap_indices] = Hs_gap_pred

final_series = reconstructed.copy()
final_series.name = "Hs_reconstructed_TCN"

if observed_mask.any():
    maximum_observation_difference = np.nanmax(
        np.abs(
            final_series.loc[observed_mask].values
            - df.loc[observed_mask, "Hs_observed"].values
        )
    )
else:
    maximum_observation_difference = np.nan

print(
    "Maximum difference in original observations:",
    maximum_observation_difference
)
print(
    "Reconstructed timestamps:",
    int(final_series.loc[df["Hs_observed"].isna()].notna().sum())
)


# =========================================
# 11) PLOTS
# =========================================
plt.figure(figsize=(13, 4))
plt.plot(
    df.index,
    df["Hs_reference"],
    label="Reference series",
    alpha=0.7,
    linewidth=1
)
plt.plot(
    final_series.index,
    final_series,
    label="TCN reconstructed series",
    linewidth=1.5
)
plt.title("Significant wave height reconstruction using TCN")
plt.xlabel("Time")
plt.ylabel(r"$H_s$ (m)")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

plt.figure(figsize=(13, 4))
plt.plot(
    final_series.index,
    final_series,
    label="TCN reconstructed series",
    linewidth=1.5,
    color="black"
)
plt.plot(
    df.index,
    df["Hs_observed"],
    label="Observed series",
    linewidth=1,
    alpha=0.7,
    color="red"
)
plt.title("Observed and reconstructed significant wave height")
plt.xlabel("Time")
plt.ylabel(r"$H_s$ (m)")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# =========================================
# SAVE FINAL TCN TIME SERIES
# =========================================

from pathlib import Path

OUTPUT_DIR = Path("model_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

output_file = OUTPUT_DIR / "M1_reconstructed_TCN.csv"

output_df = pd.DataFrame(
    {
        "Hs": final_series,
        "source": np.where(
            df["Hs_observed"].notna(),
            "observed",
            np.where(
                final_series.notna(),
                "reconstructed",
                "unavailable"
            )
        )
    },
    index=final_series.index
)

output_df.index.name = "datetime"

output_df.to_csv(
    output_file,
    index=True
)

print(f"CSV saved successfully: {output_file}")
print("Rows saved:", len(output_df))
print("\nSource counts:")
print(output_df["source"].value_counts())

In [ ]:
# =========================================
# TCN PLOT 1 + SVG EXPORT
# Gappy observed series by subset vs reconstruction
# =========================================
from pathlib import Path

save_dir = Path(OUTPUT_DIR)
save_dir.mkdir(parents=True, exist_ok=True)

# Use the final reconstructed series generated by the TCN model
series_to_plot = final_series.copy()

plt.figure(figsize=(13, 4))

plt.plot(
    series_to_plot.index,
    series_to_plot,
    label="Reconstructed series (TCN)",
    linewidth=2,
    color="black"
)

# Global temporal split masks
mask_train = df.index <= data["t_train_end"]
mask_validation = (
    (df.index > data["t_train_end"]) &
    (df.index <= data["t_val_end"])
)
mask_test = df.index > data["t_val_end"]

# Observed series split by subset
Hs_train_gappy = df["Hs_observed"].where(mask_train)
Hs_validation_gappy = df["Hs_observed"].where(mask_validation)
Hs_test_gappy = df["Hs_observed"].where(mask_test)

if Hs_train_gappy.notna().any():
    plt.plot(
        df.index,
        Hs_train_gappy,
        label="Train (with gaps)",
        alpha=0.8,
        color="0.5"
    )

if Hs_validation_gappy.notna().any():
    plt.plot(
        df.index,
        Hs_validation_gappy,
        label="Validation (with gaps)",
        alpha=0.8,
        color="orange"
    )

if Hs_test_gappy.notna().any():
    plt.plot(
        df.index,
        Hs_test_gappy,
        label="Test (with gaps)",
        alpha=0.8,
        color="green"
    )

plt.title("Gappy observed series by subset versus TCN reconstruction")
plt.xlabel("Time")
plt.ylabel(r"Significant wave height $H_s$ (m)")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()

svg_path_1 = save_dir / "TCN_gappy_series_vs_reconstruction.svg"

plt.savefig(
    svg_path_1,
    format="svg",
    bbox_inches="tight"
)

plt.show()

print(f"Figure 1 saved to: {svg_path_1}")

In [ ]:
# =========================================
# TRAIN / VALIDATION / TEST SCATTER PLOTS
# TCN model
# =========================================
def inverse_residual(values_scaled, scaler):
    """Convert normalized asinh residuals back to metres."""
    values_asinh = scaler.inverse_transform(
        np.asarray(values_scaled).reshape(-1, 1)
    ).ravel()
    return np.sinh(values_asinh)


def calculate_metrics(y_true, y_pred):
    """Calculate MAE, RMSE and R² using finite values only."""
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()

    valid = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[valid]
    y_pred = y_pred[valid]

    return (
        mean_absolute_error(y_true, y_pred),
        np.sqrt(mean_squared_error(y_true, y_pred)),
        r2_score(y_true, y_pred)
    )


# Reference values aligned with each subset
reference_train = data["sim_t"][train_mask]
reference_validation = data["sim_t"][validation_mask]
reference_test = data["sim_t"][test_mask]

# Predictions in normalized residual space
train_pred_scaled = model.predict(X_train, verbose=0).ravel()
validation_pred_scaled = model.predict(X_validation, verbose=0).ravel()
test_pred_scaled = model.predict(X_test, verbose=0).ravel()

# True residuals in metres
train_residual_true = inverse_residual(y_train, sc_y)
validation_residual_true = inverse_residual(y_validation, sc_y)
test_residual_true = inverse_residual(y_test, sc_y)

# Predicted residuals in metres
train_residual_pred = inverse_residual(train_pred_scaled, sc_y)
validation_residual_pred = inverse_residual(
    validation_pred_scaled, sc_y
)
test_residual_pred = inverse_residual(test_pred_scaled, sc_y)

# Reconstruct total Hs
Hs_train_true = reference_train + train_residual_true
Hs_validation_true = reference_validation + validation_residual_true
Hs_test_true = reference_test + test_residual_true

Hs_train_pred = reference_train + train_residual_pred
Hs_validation_pred = reference_validation + validation_residual_pred
Hs_test_pred = reference_test + test_residual_pred

# Metrics
mae_train, rmse_train, r2_train = calculate_metrics(
    Hs_train_true, Hs_train_pred
)

mae_validation, rmse_validation, r2_validation = calculate_metrics(
    Hs_validation_true, Hs_validation_pred
)

mae_test, rmse_test, r2_test = calculate_metrics(
    Hs_test_true, Hs_test_pred
)

print(
    f"[TRAIN] MAE={mae_train:.3f} m | "
    f"RMSE={rmse_train:.3f} m | R²={r2_train:.3f}"
)

print(
    f"[VALIDATION] MAE={mae_validation:.3f} m | "
    f"RMSE={rmse_validation:.3f} m | R²={r2_validation:.3f}"
)

print(
    f"[TEST] MAE={mae_test:.3f} m | "
    f"RMSE={rmse_test:.3f} m | R²={r2_test:.3f}"
)


# =========================================
# HORIZONTAL SCATTER PLOTS
# =========================================
splits = [
    (
        "TRAIN",
        Hs_train_true,
        Hs_train_pred,
        mae_train,
        rmse_train,
        r2_train
    ),
    (
        "VALIDATION",
        Hs_validation_true,
        Hs_validation_pred,
        mae_validation,
        rmse_validation,
        r2_validation
    ),
    (
        "TEST",
        Hs_test_true,
        Hs_test_pred,
        mae_test,
        rmse_test,
        r2_test
    )
]

split_colors = {
    "TRAIN": "gray",
    "VALIDATION": "tab:orange",
    "TEST": "tab:green"
}

all_values = np.concatenate(
    [np.asarray(split[1]).ravel() for split in splits] +
    [np.asarray(split[2]).ravel() for split in splits]
)

all_values = all_values[np.isfinite(all_values)]

if all_values.size == 0:
    raise ValueError("No finite values are available for plotting.")

minimum_value = float(np.min(all_values))
maximum_value = float(np.max(all_values))

padding = 0.05 * max(maximum_value - minimum_value, 1.0)
lower_limit = max(0.0, minimum_value - padding)
upper_limit = maximum_value + padding

fig, axes = plt.subplots(
    1, 3,
    figsize=(18, 6),
    sharex=True,
    sharey=True
)

for ax, (
    title,
    y_true,
    y_pred,
    mae,
    rmse,
    r2
) in zip(axes, splits):

    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()

    valid = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[valid]
    y_pred = y_pred[valid]

    ax.scatter(
        y_true,
        y_pred,
        s=10,
        alpha=0.55,
        edgecolors="none",
        color=split_colors[title],
        label=title
    )

    ax.plot(
        [lower_limit, upper_limit],
        [lower_limit, upper_limit],
        linestyle="--",
        linewidth=1.5,
        color="black",
        label="1:1 line"
    )

    ax.set_title(title)
    ax.set_xlabel(r"Observed $H_s$ (m)")
    ax.set_xlim(lower_limit, upper_limit)
    ax.set_ylim(lower_limit, upper_limit)
    ax.set_aspect("equal", adjustable="box")
    ax.grid(True, linestyle="--", alpha=0.3)
    ax.legend(loc="lower right")

    ax.annotate(
        f"MAE = {mae:.2f} m\n"
        f"RMSE = {rmse:.2f} m\n"
        f"R² = {r2:.2f}",
        xy=(0.03, 0.97),
        xycoords="axes fraction",
        va="top",
        ha="left",
        bbox={
            "boxstyle": "round",
            "facecolor": "white",
            "edgecolor": "gray",
            "alpha": 0.8
        }
    )

axes[0].set_ylabel(r"Predicted $H_s$ (m)")

plt.suptitle(
    "TCN: observed versus predicted significant wave height",
    fontsize=14,
    y=1.02
)

plt.tight_layout()
plt.show()

In [ ]:
# =========================================
# ADDITIONAL TEST METRICS — TCN
# =========================================
Hs_test_true = np.asarray(Hs_test_true).ravel()
Hs_test_pred = np.asarray(Hs_test_pred).ravel()

valid_test = np.isfinite(Hs_test_true) & np.isfinite(Hs_test_pred)

Hs_test_true_valid = Hs_test_true[valid_test]
Hs_test_pred_valid = Hs_test_pred[valid_test]

if Hs_test_true_valid.size == 0:
    raise ValueError("No finite TEST values are available.")

test_error = Hs_test_pred_valid - Hs_test_true_valid

test_mae = mean_absolute_error(
    Hs_test_true_valid,
    Hs_test_pred_valid
)

test_rmse = np.sqrt(
    mean_squared_error(
        Hs_test_true_valid,
        Hs_test_pred_valid
    )
)

test_r2 = r2_score(
    Hs_test_true_valid,
    Hs_test_pred_valid
)

if Hs_test_true_valid.size > 1:
    test_r = float(
        np.corrcoef(
            Hs_test_true_valid,
            Hs_test_pred_valid
        )[0, 1]
    )
else:
    test_r = np.nan

test_bias = float(np.mean(test_error))

# Robust MAPE
denominator = np.maximum(
    np.abs(Hs_test_true_valid),
    1e-6
)

test_mape = float(
    np.mean(
        np.abs(test_error) / denominator
    ) * 100.0
)

print(
    f"[TEST] MAE={test_mae:.3f} m | "
    f"RMSE={test_rmse:.3f} m | "
    f"R²={test_r2:.3f} | "
    f"r={test_r:.3f} | "
    f"Bias={test_bias:.3f} m | "
    f"MAPE={test_mape:.1f}%"
)


# =========================================
# TEST TIME SERIES — TCN
# =========================================
test_times = pd.DatetimeIndex(times[test_mask])
test_times_valid = test_times[valid_test]

plt.figure(figsize=(14, 4))

plt.plot(
    test_times_valid,
    Hs_test_true_valid,
    label="Observed TEST values",
    color="green",
    linewidth=1.8
)

plt.plot(
    test_times_valid,
    Hs_test_pred_valid,
    label="TCN prediction",
    color="black",
    linewidth=1.6,
    alpha=0.9
)

plt.title(
    "TCN TEST time series: observed versus predicted significant wave height"
)

plt.xlabel("Time")
plt.ylabel(r"Significant wave height $H_s$ (m)")
plt.grid(True, linestyle="--", alpha=0.35)
plt.legend()

metrics_text = (
    f"MAE:  {test_mae:.2f} m\n"
    f"RMSE: {test_rmse:.2f} m\n"
    f"R²:   {test_r2:.2f}\n"
    f"r:    {test_r:.2f}\n"
    f"Bias: {test_bias:.2f} m\n"
    f"MAPE: {test_mape:.2f}%"
)

ax = plt.gca()

ax.text(
    0.01,
    0.98,
    metrics_text,
    transform=ax.transAxes,
    va="top",
    ha="left",
    bbox={
        "boxstyle": "round",
        "facecolor": "white",
        "edgecolor": "gray",
        "alpha": 0.8
    }
)

plt.tight_layout()
plt.show()

## Convolutional LSTM model

This section implements the Convolutional Long Short-Term Memory
(ConvLSTM) model used to reconstruct missing significant wave-height
observations from the public offshore buoy record.

The input sequences are reshaped into the format required by the
ConvLSTM architecture, allowing recurrent temporal learning to be
combined with convolutional operations over the structured predictor
dimension.

The same preprocessing, residual formulation, feature set,
chronological data splitting, rolling-origin validation strategy, and
evaluation metrics used for the previous models are maintained to ensure
a consistent comparison among architectures.

In [ ]:
# -*- coding: utf-8 -*-
# Gap reconstruction in Hs_observed using a true ConvLSTM architecture (ConvLSTM2D)

import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras import mixed_precision

import optuna

# =========================================================
# 0) General configuration
# =========================================================
mixed_precision.set_global_policy("float32")
tf.keras.backend.set_floatx("float32")

np.random.seed(42)
tf.random.set_seed(42)

WINDOW = 56          # 3-hour steps; 56 approximately equals 7 days
BATCH = 64
EPOCHS = 200
P_TRAIN, P_VAL = 0.70, 0.10

N_TRIALS_LOW = 5
N_TRIALS_HIGH = 3
N_FOLDS = 3

HS_MIN, HS_MAX = 0.0, 20.0
OVERSAMPLE_WEIGHT = 4.5  # weight assigned to Hs >= training P95
# =========================================================
# 1) Residual loss in metres
# =========================================================
def mae_meters_residual_loss(sc_y):
    """
    y_true and y_pred are in the normalized target space:
        delta_t = asinh(delta)
    The loss is calculated after:
        inverse standardization -> inverse asinh -> residual in metres
    """
    def loss(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.cast(y_pred, tf.float32)

        y_true_t = y_true * sc_y.scale_[0] + sc_y.mean_[0]
        y_pred_t = y_pred * sc_y.scale_[0] + sc_y.mean_[0]

        y_true_m = tf.sinh(y_true_t)
        y_pred_m = tf.sinh(y_pred_t)

        return tf.reduce_mean(tf.abs(y_true_m - y_pred_m))
    return loss

# =========================================================
# 2) Chronological temporal split
# =========================================================
def temporal_split_percent(times_index, p_train=0.70, p_val=0.10):
    times_index = pd.DatetimeIndex(times_index)

    if len(times_index) == 0:
        raise ValueError("times_index is empty.")
    if not (0 < p_train < 1 and 0 < p_val < 1 and p_train + p_val < 1):
        raise ValueError("Invalid percentages.")

    order = np.argsort(times_index.values)
    times_sorted = times_index.values[order]
    n = len(times_sorted)

    i_train_end = int(np.floor(p_train * (n - 1)))
    i_val_end = int(np.floor((p_train + p_val) * (n - 1)))

    t_train_end = pd.to_datetime(times_sorted[i_train_end])
    t_val_end = pd.to_datetime(times_sorted[i_val_end])

    m_tr = (times_index <= t_train_end)
    m_val = (times_index > t_train_end) & (times_index <= t_val_end)
    m_te = (times_index > t_val_end)

    if not (m_tr.any() and m_val.any() and m_te.any()):
        raise ValueError(
            f"Empty split. Counts -> train:{m_tr.sum()}, val:{m_val.sum()}, test:{m_te.sum()}"
        )

    return m_tr, m_val, m_te, t_train_end, t_val_end

# =========================================================
# 3) Sequence-to-one window construction
# =========================================================
def build_seq2one_feats(arr_feats, target_arr, window, seg_ids):
    X, y, t_idx = [], [], []

    for i in range(window - 1, len(arr_feats)):
        # Do not cross segments
        if seg_ids[i] != seg_ids[i - window + 1]:
            continue

        xs = arr_feats[i - window + 1:i + 1, :]

        # Discard NaNs
        if np.isnan(xs).any():
            continue
        if np.isnan(target_arr[i]):
            continue

        X.append(xs)
        y.append(target_arr[i])
        t_idx.append(i)

    return np.array(X), np.array(y).reshape(-1, 1), np.array(t_idx, dtype=np.int64)

# =========================================================
# 4) Features
# =========================================================
def build_features(df):
    out = {}
    dfx = df.copy()

    dfx["delta"] = dfx["Hs_observed"] - dfx["Hs_reference"]
    dfx["delta_t"] = np.arcsinh(dfx["delta"])

    sim_base = dfx["Hs_reference"]

    feat = pd.DataFrame(index=dfx.index)
    feat["sim"] = sim_base

    # Rolling statistics
    for h in [8, 24, 56]:
        r = sim_base.rolling(h, min_periods=h)
        feat[f"sim_max_{h}"] = r.max()
        feat[f"sim_mean_{h}"] = r.mean()
        feat[f"sim_std_{h}"] = r.std()

    # Lags
    for k in [1, 3, 8, 24, 56]:
        feat[f"sim_lag_{k}"] = sim_base.shift(k)

    # Dynamics
    feat["sim_diff_1"] = sim_base.diff(1)
    feat["sim_diff_3"] = sim_base.diff(3)
    feat["sim_diff_8"] = sim_base.diff(8)

    feat["sim_ema_8"] = sim_base.ewm(span=8, adjust=False).mean()
    feat["sim_ema_24"] = sim_base.ewm(span=24, adjust=False).mean()
    feat["sim_ema_56"] = sim_base.ewm(span=56, adjust=False).mean()


    # Seasonality
    feat["sin_doy"] = np.sin(2 * np.pi * feat.index.dayofyear / 365.25)
    feat["cos_doy"] = np.cos(2 * np.pi * feat.index.dayofyear / 365.25)
    feat["sin_hour"] = np.sin(2 * np.pi * feat.index.hour / 24.0)
    feat["cos_hour"] = np.cos(2 * np.pi * feat.index.hour / 24.0)
    feat["sin_mon"] = np.sin(2 * np.pi * (feat.index.month - 1) / 12)
    feat["cos_mon"] = np.cos(2 * np.pi * (feat.index.month - 1) / 12)

    out["df"] = dfx
    out["feat_raw"] = feat
    return out

# =========================================================
# 5) Rolling-origin folds
# =========================================================
def make_folds(times, n_folds=3):
    n = len(times)
    idx = np.linspace(int(0.55 * n), n - 1, n_folds + 1, dtype=int)

    folds = []
    for i in range(n_folds):
        t_train_end = pd.Timestamp(times[idx[i]])
        t_val_end = pd.Timestamp(times[idx[i + 1]])
        folds.append((t_train_end, t_val_end))

    return folds

# =========================================================
# 6) Fold-specific dataset without leakage
# =========================================================
def build_dataset_with_cuts(df_slice, window, t_train_end, t_val_end):
    dfl = df_slice.copy()

    if "seg" in dfl.columns:
        pass
    else:
        deltas = dfl.index.to_series().diff()
        freq = deltas.mode().iloc[0]
        tol = pd.Timedelta("5min")
        same_step = deltas.between(freq - tol, freq + tol)
        dfl["seg"] = (~same_step | dfl["Hs_reference"].isna()).cumsum().fillna(0).astype(int)

    ctx = build_features(dfl)
    feat = ctx["feat_raw"]

    dfl["delta"] = dfl["Hs_observed"] - dfl["Hs_reference"]
    dfl["delta_t"] = np.arcsinh(dfl["delta"])

    mask_scal = (dfl.index <= t_train_end)

    sc_X_f = StandardScaler().fit(feat.loc[mask_scal].values)
    sc_y_f = StandardScaler().fit(dfl.loc[mask_scal, ["delta_t"]].dropna())

    feat_sc = pd.DataFrame(
        sc_X_f.transform(feat.values),
        index=feat.index,
        columns=feat.columns
    )

    dfl["delta_scaled"] = np.nan
    ok = dfl["delta_t"].notna()
    dfl.loc[ok, "delta_scaled"] = sc_y_f.transform(dfl.loc[ok, ["delta_t"]])[:, 0]

    Xf, yf, t_idx = build_seq2one_feats(
        feat_sc.values,
        dfl["delta_scaled"].values,
        window,
        dfl["seg"].values
    )

    t_idx = np.asarray(t_idx, dtype=np.int64)
    times_f = dfl.index[t_idx]
    sim_t = dfl["Hs_reference"].values[t_idx]

    m_tr = (times_f <= t_train_end)
    m_val = (times_f > t_train_end) & (times_f <= t_val_end)

    return Xf, yf, times_f, m_tr, m_val, sc_y_f, sim_t

# =========================================================
# 7) Global dataset without leakage
# =========================================================
def build_dataset_no_leak(df_slice, window, p_train=P_TRAIN, p_val=P_VAL):
    ctx = build_features(df_slice)
    df_loc = ctx["df"]
    feat = ctx["feat_raw"]

    if "seg" in df_loc.columns:
        pass
    else:
        deltas = df_loc.index.to_series().diff()
        freq = deltas.mode().iloc[0]
        tol = pd.Timedelta("5min")
        same_step = deltas.between(freq - tol, freq + tol)
        df_loc["seg"] = (~same_step | df_loc["Hs_reference"].isna()).cumsum().fillna(0).astype(int)

    df_loc["delta"] = df_loc["Hs_observed"] - df_loc["Hs_reference"]
    df_loc["delta_t"] = np.arcsinh(df_loc["delta"])

    X0, _, t_idx0 = build_seq2one_feats(
        feat.values,
        df_loc["delta_t"].values,
        window,
        df_loc["seg"].values
    )

    t_idx0 = np.asarray(t_idx0, dtype=np.int64)
    times0 = df_loc.index[t_idx0]
    _, _, _, t_train_end, t_val_end = temporal_split_percent(times0, p_train, p_val)

    mask_scal = (df_loc.index <= t_train_end)

    sc_X = StandardScaler().fit(feat.loc[mask_scal].values)
    sc_y = StandardScaler().fit(df_loc.loc[mask_scal, ["delta_t"]].dropna())

    feat_sc = pd.DataFrame(
        sc_X.transform(feat.values),
        index=feat.index,
        columns=feat.columns
    )

    df_loc["delta_scaled"] = np.nan
    ok = df_loc["delta_t"].notna()
    df_loc.loc[ok, "delta_scaled"] = sc_y.transform(df_loc.loc[ok, ["delta_t"]])[:, 0]

    X, y, t_idx = build_seq2one_feats(
        feat_sc.values,
        df_loc["delta_scaled"].values,
        window,
        df_loc["seg"].values
    )

    t_idx = np.asarray(t_idx, dtype=np.int64)
    times = df_loc.index[t_idx]
    sim_t = df_loc["Hs_reference"].values[t_idx]

    m_tr, m_val, m_te, t_train_end, t_val_end = temporal_split_percent(times, p_train, p_val)

    return {
        "X": X,
        "y": y,
        "times": times,
        "m_tr": m_tr,
        "m_val": m_val,
        "m_te": m_te,
        "t_train_end": t_train_end,
        "t_val_end": t_val_end,
        "sc_X": sc_X,
        "sc_y": sc_y,
        "feat_scaled": feat_sc,
        "df_loc": df_loc,
        "sim_t": sim_t
    }

# =========================================================
# 8) True ConvLSTM model
# =========================================================
def reshape_for_convlstm(X):
    """
    ConvLSTM2D expects:
    (batch, time, rows, cols, channels)

    The input is reshaped from:
    (N, WINDOW, n_feats)
    ->
    (N, WINDOW, n_feats, 1, 1)
    """
    return X[..., np.newaxis, np.newaxis].astype("float32")

def build_true_convlstm_model(
    window,
    n_feats,
    filters=32,
    kernel_size=(3, 1),
    dropout=0.20,
    recurrent_dropout=0.10,
    dense_units=16,
    l2reg=0.0,
    lr=3e-4,
    use_adamw=False,
    use_second_layer=False
):
    reg = tf.keras.regularizers.l2(l2reg) if (l2reg and l2reg > 0) else None

    inp = layers.Input(shape=(window, n_feats, 1, 1))

    x = layers.ConvLSTM2D(
        filters=filters,
        kernel_size=kernel_size,
        padding="same",
        return_sequences=use_second_layer,
        activation="tanh",
        recurrent_activation="sigmoid",
        dropout=dropout,
        recurrent_dropout=recurrent_dropout,
        kernel_regularizer=reg,
        recurrent_regularizer=reg
    )(inp)

    if use_second_layer:
        x = layers.ConvLSTM2D(
            filters=filters,
            kernel_size=kernel_size,
            padding="same",
            return_sequences=False,
            activation="tanh",
            recurrent_activation="sigmoid",
            dropout=dropout,
            recurrent_dropout=recurrent_dropout,
            kernel_regularizer=reg,
            recurrent_regularizer=reg
        )(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(dense_units, activation="relu", kernel_regularizer=reg)(x)
    out = layers.Dense(1, kernel_regularizer=reg)(x)

    model = Model(inp, out)

    opt = (
        tf.keras.optimizers.AdamW(learning_rate=lr, weight_decay=l2reg, clipnorm=1.0)
        if use_adamw and hasattr(tf.keras.optimizers, "AdamW")
        else tf.keras.optimizers.Adam(learning_rate=lr, clipnorm=1.0)
    )

    model.compile(optimizer=opt, loss="mae")
    return model

# =========================================================
# 9) Initial data loading and preprocessing
# =========================================================
# df_all must already be loaded with:
# a temporal index
# columns: Hs_observed, Hs_reference

df = df_all.loc["1990-01-01":"2026-01-01", ["Hs_observed", "Hs_reference"]].copy().sort_index()
df = df.resample("3h").mean()


# Segmentation
deltas = df.index.to_series().diff()
freq = deltas.mode().iloc[0]
tol = pd.Timedelta("5min")
same_step = deltas.between(freq - tol, freq + tol)
df["seg"] = (~same_step | df["Hs_reference"].isna()).cumsum().fillna(0).astype(int)

# =========================================================
# 10) CONVLSTM HYPERPARAMETER OPTIMIZATION
# =========================================================
import gc
from optuna.trial import TrialState

windows = [24, 32, 40, 48, 56, 72]

# External boundary: TEST remains completely outside HPO
cache = {
    W: build_dataset_no_leak(df, W, p_train=P_TRAIN, p_val=P_VAL)
    for W in windows
}
HPO_END_TIME = min(pd.Timestamp(cache[W]["t_val_end"]) for W in windows)
df_hpo = df.loc[df.index <= HPO_END_TIME].copy()
future_times = df.index[df.index > HPO_END_TIME]

print("\n=========================================")
print("EXTERNAL HPO SEPARATION — CONVLSTM")
print("=========================================")
print("Last timestamp available to Optuna:", HPO_END_TIME)
print("First timestamp reserved outside HPO:", future_times.min())
print("HPO timestamps:", len(df_hpo))
print("Timestamps after HPO:", len(future_times))


def get_hpo_folds(df_hpo_slice, window, n_folds):
    """Generate rolling-origin folds exclusively inside the HPO period."""
    ctx = build_features(df_hpo_slice)
    _, _, idx = build_seq2one_feats(
        ctx["feat_raw"].values,
        ctx["df"]["delta_t"].values,
        window,
        ctx["df"]["seg"].values
    )
    idx = np.asarray(idx, dtype=np.int64)

    if idx.size == 0:
        raise ValueError(f"No valid windows generated for WINDOW={window}.")

    return make_folds(ctx["df"].index[idx].to_numpy(), n_folds=n_folds)


def objective(trial, budget="low"):
    print(f"\n>>> ConvLSTM trial {trial.number + 1} ({budget}) started")

    window_trial = trial.suggest_categorical("WINDOW", windows)
    filters = trial.suggest_categorical("filters", [16, 32, 48])
    k_time = trial.suggest_categorical("k_time", [3, 5])
    dropout = trial.suggest_float("dropout", 0.15, 0.40, step=0.05)
    recurrent_dropout = trial.suggest_float(
        "recurrent_dropout", 0.00, 0.30, step=0.05
    )
    dense_units = trial.suggest_categorical("dense_units", [8, 16, 32])
    l2reg = trial.suggest_float("l2reg", 0.0, 1e-4, step=1e-5)
    learning_rate = trial.suggest_categorical(
        "lr", [1e-3, 5e-4, 3e-4, 2e-4, 1e-4]
    )
    batch_size = trial.suggest_categorical(
        "batch_size", [32, 64, 128, 256]
    )
    use_adamw = trial.suggest_categorical("use_adamw", [True, False])
    use_second_layer = trial.suggest_categorical(
        "use_second_layer", [False, True]
    )

    epochs_trial = 30 if budget == "low" else 80
    n_folds_trial = 2 if budget == "low" else N_FOLDS
    folds = get_hpo_folds(df_hpo, window_trial, n_folds_trial)

    validation_rmse = []
    training_rmse = []
    parameter_counts = []

    for fold_i, (train_end, validation_end) in enumerate(folds, start=1):
        print(
            f"    Fold {fold_i}/{len(folds)} | "
            f"train <= {train_end} | validation <= {validation_end}"
        )

        (
            X_fold, y_fold, times_fold, train_mask, validation_mask,
            sc_y_fold, reference_target
        ) = build_dataset_with_cuts(
            df_hpo, window_trial, train_end, validation_end
        )

        if not train_mask.any() or not validation_mask.any():
            raise ValueError(f"Fold {fold_i} contains an empty split.")

        X_train_fold = X_fold[train_mask].astype("float32")
        y_train_fold = y_fold[train_mask].astype("float32")
        X_validation_fold = X_fold[validation_mask].astype("float32")
        y_validation_fold = y_fold[validation_mask].astype("float32")

        reference_train = reference_target[train_mask]
        reference_validation = reference_target[validation_mask]

        # Extreme-event weights calculated from training only
        residual_train = np.sinh(
            sc_y_fold.inverse_transform(y_train_fold)[:, 0]
        )
        Hs_train_true = reference_train + residual_train
        p95 = np.percentile(Hs_train_true, 95)

        sample_weights = np.ones_like(Hs_train_true, dtype=np.float32)
        sample_weights[Hs_train_true >= p95] = float(OVERSAMPLE_WEIGHT)

        X_train_conv = reshape_for_convlstm(X_train_fold)
        X_validation_conv = reshape_for_convlstm(X_validation_fold)

        tf.keras.backend.clear_session()

        fold_model = build_true_convlstm_model(
            window=window_trial,
            n_feats=X_fold.shape[-1],
            filters=filters,
            kernel_size=(k_time, 1),
            dropout=dropout,
            recurrent_dropout=recurrent_dropout,
            dense_units=dense_units,
            l2reg=l2reg,
            lr=learning_rate,
            use_adamw=use_adamw,
            use_second_layer=use_second_layer
        )

        fold_model.compile(
            optimizer=fold_model.optimizer,
            loss=mae_meters_residual_loss(sc_y_fold)
        )

        fold_history = fold_model.fit(
            X_train_conv,
            y_train_fold,
            validation_data=(X_validation_conv, y_validation_fold),
            sample_weight=sample_weights,
            epochs=epochs_trial,
            batch_size=batch_size,
            verbose=0,
            callbacks=[
                EarlyStopping(
                    monitor="val_loss",
                    patience=4,
                    min_delta=1e-4,
                    restore_best_weights=True
                ),
                ReduceLROnPlateau(
                    monitor="val_loss",
                    patience=2,
                    factor=0.5,
                    min_lr=1e-5
                )
            ]
        )

        for epoch_i, val_loss in enumerate(fold_history.history["val_loss"]):
            step = (fold_i - 1) * epochs_trial + epoch_i
            trial.report(float(val_loss), step=step)

            if trial.should_prune():
                del fold_model
                tf.keras.backend.clear_session()
                gc.collect()
                raise optuna.exceptions.TrialPruned()

        # Validation RMSE
        validation_pred_scaled = fold_model.predict(
            X_validation_conv, verbose=0
        ).ravel()
        validation_pred_residual = np.sinh(
            sc_y_fold.inverse_transform(
                validation_pred_scaled.reshape(-1, 1)
            ).ravel()
        )
        Hs_validation_pred = (
            reference_validation + validation_pred_residual
        )

        validation_true_residual = np.sinh(
            sc_y_fold.inverse_transform(y_validation_fold).ravel()
        )
        Hs_validation_true = (
            reference_validation + validation_true_residual
        )

        rmse_validation = np.sqrt(
            mean_squared_error(Hs_validation_true, Hs_validation_pred)
        )
        validation_rmse.append(float(rmse_validation))

        # Training RMSE
        train_pred_scaled = fold_model.predict(
            X_train_conv, verbose=0
        ).ravel()
        train_pred_residual = np.sinh(
            sc_y_fold.inverse_transform(
                train_pred_scaled.reshape(-1, 1)
            ).ravel()
        )
        Hs_train_pred = reference_train + train_pred_residual

        train_true_residual = np.sinh(
            sc_y_fold.inverse_transform(y_train_fold).ravel()
        )
        Hs_train_true = reference_train + train_true_residual

        rmse_train = np.sqrt(
            mean_squared_error(Hs_train_true, Hs_train_pred)
        )
        training_rmse.append(float(rmse_train))

        parameter_counts.append(
            int(np.sum([
                np.prod(variable.shape)
                for variable in fold_model.trainable_weights
            ]))
        )

        print(
            f"        RMSE train={rmse_train:.4f} m | "
            f"RMSE validation={rmse_validation:.4f} m"
        )

        del fold_model
        tf.keras.backend.clear_session()
        gc.collect()

    mean_val_rmse = float(np.mean(validation_rmse))
    mean_train_rmse = float(np.mean(training_rmse))
    val_rmse_std = float(np.std(validation_rmse))
    ratio = mean_val_rmse / max(mean_train_rmse, 1e-6)
    gap = max(0.0, mean_val_rmse - mean_train_rmse)
    n_parameters = int(np.mean(parameter_counts))

    objective_value = (
        mean_val_rmse
        + 0.75 * gap
        + 0.25 * max(0.0, ratio - 1.0)
        + 1e-7 * n_parameters
        + 0.25 * val_rmse_std
    )

    trial.set_user_attr("budget", budget)
    trial.set_user_attr("mean_validation_rmse", mean_val_rmse)
    trial.set_user_attr("mean_training_rmse", mean_train_rmse)
    trial.set_user_attr("validation_rmse_std", val_rmse_std)
    trial.set_user_attr("generalization_gap", gap)
    trial.set_user_attr("parameter_count", n_parameters)

    print(
        f">>> ConvLSTM trial {trial.number + 1} completed | "
        f"validation={mean_val_rmse:.3f} m | "
        f"train={mean_train_rmse:.3f} m | "
        f"gap={gap:.3f} m | objective={objective_value:.3f}"
    )

    return objective_value


# =========================================================
# 11) LOW-BUDGET PHASE
# =========================================================
study_low = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(
        seed=42, multivariate=True, group=True
    ),
    pruner=optuna.pruners.HyperbandPruner(
        min_resource=10, max_resource=30, reduction_factor=3
    ),
    study_name="ConvLSTM_low_budget"
)

study_low.optimize(
    lambda trial: objective(trial, budget="low"),
    n_trials=N_TRIALS_LOW,
    gc_after_trial=True
)

completed_low = sorted(
    [
        trial for trial in study_low.trials
        if trial.state == TrialState.COMPLETE and trial.value is not None
    ],
    key=lambda trial: trial.value
)

if not completed_low:
    raise RuntimeError("No low-budget ConvLSTM trial completed successfully.")

top_low_trials = completed_low[:min(5, len(completed_low))]

print("\nBEST LOW-BUDGET CONVLSTM CANDIDATES")
for rank, trial in enumerate(top_low_trials, start=1):
    print(
        f"{rank}. Trial {trial.number} | "
        f"objective={trial.value:.4f} | params={trial.params}"
    )


# =========================================================
# 12) HIGH-BUDGET PHASE
# =========================================================
study_high = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(
        seed=42, multivariate=True, group=True
    ),
    pruner=optuna.pruners.HyperbandPruner(
        min_resource=10, max_resource=80, reduction_factor=3
    ),
    study_name="ConvLSTM_high_budget"
)

for candidate in top_low_trials:
    study_high.enqueue_trial(candidate.params)

study_high.optimize(
    lambda trial: objective(trial, budget="high"),
    n_trials=max(N_TRIALS_HIGH, len(top_low_trials)),
    gc_after_trial=True
)

completed_high = [
    trial for trial in study_high.trials
    if trial.state == TrialState.COMPLETE and trial.value is not None
]

if not completed_high:
    raise RuntimeError("No high-budget ConvLSTM trial completed successfully.")

best_trial = min(completed_high, key=lambda trial: trial.value)
best = best_trial.params

print("\n=========================================")
print("BEST HIGH-BUDGET CONVLSTM CONFIGURATION")
print("=========================================")
print("Best objective:", best_trial.value)
print("Best hyperparameters:", best)
print("Mean validation RMSE:", best_trial.user_attrs["mean_validation_rmse"])
print("Mean training RMSE:", best_trial.user_attrs["mean_training_rmse"])
print("Generalization gap:", best_trial.user_attrs["generalization_gap"])


# =========================================================
# 13) FINAL CONVLSTM TRAINING
# =========================================================
WINDOW = best.get("WINDOW", WINDOW)
BATCH = best.get("batch_size", BATCH)

data = build_dataset_no_leak(df, WINDOW, P_TRAIN, P_VAL)

X, y, times = data["X"], data["y"], data["times"]
train_mask = data["m_tr"]
validation_mask = data["m_val"]
test_mask = data["m_te"]
sc_y = data["sc_y"]
feat_scaled = data["feat_scaled"]

X_train = X[train_mask].astype("float32")
y_train = y[train_mask].astype("float32")
X_validation = X[validation_mask].astype("float32")
y_validation = y[validation_mask].astype("float32")
X_test = X[test_mask].astype("float32")
y_test = y[test_mask].astype("float32")

print("\n=========================================")
print("FINAL CONVLSTM TEMPORAL SPLIT")
print("=========================================")
print("Train:", X_train.shape)
print("Validation:", X_validation.shape)
print("Test:", X_test.shape)
print("End of training:", data["t_train_end"])
print("End of validation:", data["t_val_end"])
print("Start of test:", times[test_mask].min())

# Extreme-event weighting derived from training only

residual_train = np.sinh(sc_y.inverse_transform(y_train)[:, 0])
Hs_train = data["sim_t"][train_mask] + residual_train
p95 = np.percentile(Hs_train, 95)

training_weights = np.ones_like(Hs_train, dtype=np.float32)
training_weights[Hs_train >= p95] = float(OVERSAMPLE_WEIGHT)

X_train_conv = reshape_for_convlstm(X_train)
X_validation_conv = reshape_for_convlstm(X_validation)
X_test_conv = reshape_for_convlstm(X_test)

tf.keras.backend.clear_session()

model = build_true_convlstm_model(
    window=WINDOW,
    n_feats=X.shape[-1],
    filters=best.get("filters", 32),
    kernel_size=(best.get("k_time", 3), 1),
    dropout=best.get("dropout", 0.20),
    recurrent_dropout=best.get("recurrent_dropout", 0.10),
    dense_units=best.get("dense_units", 16),
    l2reg=best.get("l2reg", 0.0),
    lr=best.get("lr", 3e-4),
    use_adamw=best.get("use_adamw", False),
    use_second_layer=best.get("use_second_layer", False)
)

model.compile(
    optimizer=model.optimizer,
    loss=mae_meters_residual_loss(sc_y)
)

history = model.fit(
    X_train_conv,
    y_train,
    validation_data=(X_validation_conv, y_validation),
    sample_weight=training_weights,
    epochs=EPOCHS,
    batch_size=BATCH,
    verbose=0,
    callbacks=[
        EarlyStopping(
            monitor="val_loss",
            patience=8,
            min_delta=1e-4,
            restore_best_weights=True
        ),
        ReduceLROnPlateau(
            monitor="val_loss",
            patience=4,
            factor=0.5,
            min_lr=1e-6
        )
    ]
)

print("Best validation loss:", float(np.min(history.history["val_loss"])))


# =========================================================
# 14) EXTERNAL TEST
# =========================================================
test_pred_scaled = model.predict(X_test_conv, verbose=0).ravel()
test_pred_residual = np.sinh(
    sc_y.inverse_transform(test_pred_scaled.reshape(-1, 1)).ravel()
)
Hs_test_pred = data["sim_t"][test_mask] + test_pred_residual

test_true_residual = np.sinh(
    sc_y.inverse_transform(y_test).ravel()
)
Hs_test_true = data["sim_t"][test_mask] + test_true_residual

test_mae = mean_absolute_error(Hs_test_true, Hs_test_pred)
test_rmse = np.sqrt(mean_squared_error(Hs_test_true, Hs_test_pred))
test_r2 = r2_score(Hs_test_true, Hs_test_pred)

print("\n=========================================")
print("EXTERNAL CONVLSTM TEST RESULTS")
print("=========================================")
print(
    f"MAE={test_mae:.3f} m | "
    f"RMSE={test_rmse:.3f} m | "
    f"R²={test_r2:.3f}"
)


# =========================================================
# 15) TARGETED GAP RECONSTRUCTION
# =========================================================
reconstructed = pd.Series(
    np.nan, index=df.index, name="Hs_reconstructed"
)

# Preserve original observations exactly
observed_mask = df["Hs_observed"].notna()
reconstructed.loc[observed_mask] = df.loc[
    observed_mask, "Hs_observed"
].values

missing_indices = np.where(df["Hs_observed"].isna())[0]
scaled_features = feat_scaled.values
segments = df["seg"].values

valid_missing_indices = [
    k for k in missing_indices
    if (
        k >= WINDOW - 1
        and segments[k] == segments[k - WINDOW + 1]
        and not np.isnan(
            scaled_features[k - WINDOW + 1:k + 1]
        ).any()
    )
]

print("\n=========================================")
print("CONVLSTM GAP RECONSTRUCTION")
print("=========================================")
print("Total missing timestamps:", len(missing_indices))
print("Valid reconstruction windows:", len(valid_missing_indices))

if valid_missing_indices:
    X_gap = np.stack(
        [
            scaled_features[k - WINDOW + 1:k + 1]
            for k in valid_missing_indices
        ],
        axis=0
    ).astype("float32")

    X_gap_conv = reshape_for_convlstm(X_gap)

    gap_pred_scaled = model.predict(X_gap_conv, verbose=0).ravel()
    gap_pred_residual = np.sinh(
        sc_y.inverse_transform(
            gap_pred_scaled.reshape(-1, 1)
        ).ravel()
    )

    reference_at_gap = df["Hs_reference"].values[valid_missing_indices]
    Hs_gap_pred = np.clip(
        reference_at_gap + gap_pred_residual,
        HS_MIN,
        HS_MAX
    )

    reconstructed.iloc[valid_missing_indices] = Hs_gap_pred

final_series = reconstructed.copy()
final_series.name = "Hs_reconstructed_ConvLSTM"

if observed_mask.any():
    maximum_observation_difference = np.nanmax(
        np.abs(
            final_series.loc[observed_mask].values
            - df.loc[observed_mask, "Hs_observed"].values
        )
    )
else:
    maximum_observation_difference = np.nan

print(
    "Maximum difference in original observations:",
    maximum_observation_difference
)
print(
    "Reconstructed timestamps:",
    int(final_series.loc[df["Hs_observed"].isna()].notna().sum())
)


# =========================================================
# 16) PLOTS
# =========================================================
plt.figure(figsize=(13, 4))
plt.plot(
    df.index,
    df["Hs_reference"],
    label="Reference series",
    alpha=0.7,
    linewidth=1
)
plt.plot(
    final_series.index,
    final_series,
    label="ConvLSTM reconstructed series",
    linewidth=1.5
)
plt.title("Significant wave height reconstruction using ConvLSTM")
plt.xlabel("Time")
plt.ylabel(r"$H_s$ (m)")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

plt.figure(figsize=(13, 4))
plt.plot(
    final_series.index,
    final_series,
    label="ConvLSTM reconstructed series",
    linewidth=1.5,
    color="black"
)
plt.plot(
    df.index,
    df["Hs_observed"],
    label="Observed series",
    linewidth=1,
    alpha=0.7,
    color="red"
)
plt.title("Observed and reconstructed significant wave height")
plt.xlabel("Time")
plt.ylabel(r"$H_s$ (m)")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# =========================================
# SAVE FINAL CONVLSTM TIME SERIES
# =========================================

from pathlib import Path

OUTPUT_DIR = Path("model_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

output_file = OUTPUT_DIR / "M1_reconstructed_ConvLSTM.csv"

output_df = pd.DataFrame(
    {
        "Hs": final_series,
        "source": np.where(
            df["Hs_observed"].notna(),
            "observed",
            np.where(
                final_series.notna(),
                "reconstructed",
                "unavailable"
            )
        )
    },
    index=final_series.index
)

output_df.index.name = "datetime"

output_df.to_csv(
    output_file,
    index=True
)

print(f"CSV saved successfully: {output_file}")
print("Rows saved:", len(output_df))
print("\nSource counts:")
print(output_df["source"].value_counts())

In [ ]:
# =========================================
# TRAIN / VALIDATION / TEST SCATTER PLOTS
# ConvLSTM model
# =========================================
def inverse_residual(values_scaled, scaler):
    """Convert normalized asinh residuals back to metres."""
    values_asinh = scaler.inverse_transform(
        np.asarray(values_scaled).reshape(-1, 1)
    ).ravel()
    return np.sinh(values_asinh)


def calculate_metrics(y_true, y_pred):
    """Calculate MAE, RMSE and R² using finite values only."""
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()

    valid = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[valid]
    y_pred = y_pred[valid]

    return (
        mean_absolute_error(y_true, y_pred),
        np.sqrt(mean_squared_error(y_true, y_pred)),
        r2_score(y_true, y_pred)
    )


# Reference values aligned with each subset
reference_train = data["sim_t"][train_mask]
reference_validation = data["sim_t"][validation_mask]
reference_test = data["sim_t"][test_mask]

# Predictions in normalized residual space
train_pred_scaled = model.predict(
    X_train_conv,
    verbose=0
).ravel()

validation_pred_scaled = model.predict(
    X_validation_conv,
    verbose=0
).ravel()

test_pred_scaled = model.predict(
    X_test_conv,
    verbose=0
).ravel()

# True residuals in metres
train_residual_true = inverse_residual(
    y_train,
    sc_y
)

validation_residual_true = inverse_residual(
    y_validation,
    sc_y
)

test_residual_true = inverse_residual(
    y_test,
    sc_y
)

# Predicted residuals in metres
train_residual_pred = inverse_residual(
    train_pred_scaled,
    sc_y
)

validation_residual_pred = inverse_residual(
    validation_pred_scaled,
    sc_y
)

test_residual_pred = inverse_residual(
    test_pred_scaled,
    sc_y
)

# Reconstruct total Hs = reference + residual
Hs_train_true = reference_train + train_residual_true
Hs_validation_true = reference_validation + validation_residual_true
Hs_test_true = reference_test + test_residual_true

Hs_train_pred = reference_train + train_residual_pred
Hs_validation_pred = reference_validation + validation_residual_pred
Hs_test_pred = reference_test + test_residual_pred

# Metrics
mae_train, rmse_train, r2_train = calculate_metrics(
    Hs_train_true,
    Hs_train_pred
)

mae_validation, rmse_validation, r2_validation = calculate_metrics(
    Hs_validation_true,
    Hs_validation_pred
)

mae_test, rmse_test, r2_test = calculate_metrics(
    Hs_test_true,
    Hs_test_pred
)

print(
    f"[TRAIN] MAE={mae_train:.3f} m | "
    f"RMSE={rmse_train:.3f} m | "
    f"R²={r2_train:.3f}"
)

print(
    f"[VALIDATION] MAE={mae_validation:.3f} m | "
    f"RMSE={rmse_validation:.3f} m | "
    f"R²={r2_validation:.3f}"
)

print(
    f"[TEST] MAE={mae_test:.3f} m | "
    f"RMSE={rmse_test:.3f} m | "
    f"R²={r2_test:.3f}"
)


# =========================================
# HORIZONTAL SCATTER PLOTS
# =========================================
splits = [
    (
        "TRAIN",
        Hs_train_true,
        Hs_train_pred,
        mae_train,
        rmse_train,
        r2_train
    ),
    (
        "VALIDATION",
        Hs_validation_true,
        Hs_validation_pred,
        mae_validation,
        rmse_validation,
        r2_validation
    ),
    (
        "TEST",
        Hs_test_true,
        Hs_test_pred,
        mae_test,
        rmse_test,
        r2_test
    )
]

split_colors = {
    "TRAIN": "gray",
    "VALIDATION": "tab:orange",
    "TEST": "tab:green"
}

all_values = np.concatenate(
    [np.asarray(split[1]).ravel() for split in splits] +
    [np.asarray(split[2]).ravel() for split in splits]
)

all_values = all_values[np.isfinite(all_values)]

if all_values.size == 0:
    raise ValueError("No finite values are available for plotting.")

minimum_value = float(np.min(all_values))
maximum_value = float(np.max(all_values))

padding = 0.05 * max(
    maximum_value - minimum_value,
    1.0
)

lower_limit = max(
    0.0,
    minimum_value - padding
)

upper_limit = maximum_value + padding

fig, axes = plt.subplots(
    1,
    3,
    figsize=(18, 6),
    sharex=True,
    sharey=True
)

for ax, (
    title,
    y_true,
    y_pred,
    mae,
    rmse,
    r2
) in zip(axes, splits):

    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()

    valid = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true_valid = y_true[valid]
    y_pred_valid = y_pred[valid]

    ax.scatter(
        y_true_valid,
        y_pred_valid,
        s=10,
        alpha=0.55,
        edgecolors="none",
        color=split_colors[title],
        label=title
    )

    ax.plot(
        [lower_limit, upper_limit],
        [lower_limit, upper_limit],
        linestyle="--",
        linewidth=1.5,
        color="black",
        label="1:1 line"
    )

    ax.set_title(title)
    ax.set_xlabel(r"Observed $H_s$ (m)")
    ax.set_xlim(lower_limit, upper_limit)
    ax.set_ylim(lower_limit, upper_limit)
    ax.set_aspect("equal", adjustable="box")
    ax.grid(True, linestyle="--", alpha=0.3)
    ax.legend(loc="lower right")

    ax.annotate(
        f"MAE = {mae:.2f} m\n"
        f"RMSE = {rmse:.2f} m\n"
        f"R² = {r2:.2f}",
        xy=(0.03, 0.97),
        xycoords="axes fraction",
        va="top",
        ha="left",
        bbox={
            "boxstyle": "round",
            "facecolor": "white",
            "edgecolor": "gray",
            "alpha": 0.8
        }
    )

axes[0].set_ylabel(r"Predicted $H_s$ (m)")

plt.suptitle(
    "ConvLSTM: observed versus predicted significant wave height",
    fontsize=14,
    y=1.02
)

plt.tight_layout()
plt.show()

In [ ]:
# =========================================
# ADDITIONAL TEST METRICS — CONVLSTM
# =========================================
Hs_test_true = np.asarray(Hs_test_true).ravel()
Hs_test_pred = np.asarray(Hs_test_pred).ravel()

valid_test = np.isfinite(Hs_test_true) & np.isfinite(Hs_test_pred)

Hs_test_true_valid = Hs_test_true[valid_test]
Hs_test_pred_valid = Hs_test_pred[valid_test]

if Hs_test_true_valid.size == 0:
    raise ValueError("No finite TEST values are available.")

test_error = Hs_test_pred_valid - Hs_test_true_valid

test_mae = mean_absolute_error(
    Hs_test_true_valid,
    Hs_test_pred_valid
)

test_rmse = np.sqrt(
    mean_squared_error(
        Hs_test_true_valid,
        Hs_test_pred_valid
    )
)

test_r2 = r2_score(
    Hs_test_true_valid,
    Hs_test_pred_valid
)

if Hs_test_true_valid.size > 1:
    test_r = float(
        np.corrcoef(
            Hs_test_true_valid,
            Hs_test_pred_valid
        )[0, 1]
    )
else:
    test_r = np.nan

test_bias = float(np.mean(test_error))

# Robust MAPE
denominator = np.maximum(
    np.abs(Hs_test_true_valid),
    1e-6
)

test_mape = float(
    np.mean(
        np.abs(test_error) / denominator
    ) * 100.0
)

print(
    f"[TEST] MAE={test_mae:.3f} m | "
    f"RMSE={test_rmse:.3f} m | "
    f"R²={test_r2:.3f} | "
    f"r={test_r:.3f} | "
    f"Bias={test_bias:.3f} m | "
    f"MAPE={test_mape:.1f}%"
)


# =========================================
# TEST TIME SERIES — CONVLSTM
# =========================================
test_times = pd.DatetimeIndex(times[test_mask])
test_times_valid = test_times[valid_test]

plt.figure(figsize=(14, 4))

plt.plot(
    test_times_valid,
    Hs_test_true_valid,
    label="Observed TEST values",
    color="green",
    linewidth=1.8
)

plt.plot(
    test_times_valid,
    Hs_test_pred_valid,
    label="ConvLSTM prediction",
    linewidth=1.6,
    alpha=0.9,
    color="orchid"
)

plt.title(
    "ConvLSTM TEST time series: observed versus predicted significant wave height"
)

plt.xlabel("Time")
plt.ylabel(r"Significant wave height $H_s$ (m)")
plt.grid(True, linestyle="--", alpha=0.35)
plt.legend()

metrics_text = (
    f"MAE:  {test_mae:.2f} m\n"
    f"RMSE: {test_rmse:.2f} m\n"
    f"R²:   {test_r2:.2f}\n"
    f"r:    {test_r:.2f}\n"
    f"Bias: {test_bias:.2f} m\n"
    f"MAPE: {test_mape:.2f}%"
)

ax = plt.gca()

ax.text(
    0.01,
    0.98,
    metrics_text,
    transform=ax.transAxes,
    va="top",
    ha="left",
    bbox={
        "boxstyle": "round",
        "facecolor": "white",
        "edgecolor": "gray",
        "alpha": 0.8
    }
)

plt.tight_layout()
plt.show()